In [1]:
import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')  
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

<Client: 'tcp://203.247.189.225:39418' processes=5 threads=90, memory=419.10 GiB>

In [2]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp
import xcesm

In [3]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_preprocessing import CESM2_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

In [4]:
# lim_varnames = [
#     "sp_Fe_lim_Cweight_avg_100m",
#     "sp_N_lim_Cweight_avg_100m",
#     "sp_P_lim_Cweight_avg_100m",
#     "diat_Fe_lim_Cweight_avg_100m",
#     "diat_N_lim_Cweight_avg_100m",
#     "diat_P_lim_Cweight_avg_100m",
#     "diat_SiO3_lim_Cweight_avg_100m",
# ]

lim_varnames_2d = [
    "TAUX",
    "TAUY",
    "sp_Fe_lim_surf",
    "sp_N_lim_surf",
    "sp_P_lim_surf",
    "diat_Fe_lim_surf",
    "diat_N_lim_surf",
    "diat_P_lim_surf",
    "diat_SiO3_lim_surf",
    "HBLT",
    "SSH",
    "BSF",   
    "LWDN_F",
    "graze_diat_zoo_zint_100m",
    "graze_diat_zint_100m",
    "zoo_loss_zint_100m",
    "IFRAC",
    "sp_light_lim_surf",
    "diat_light_lim_surf",
    "diat_agg_zint_100m",
    "sp_agg_zint_100m",
    "photoC_diat_zint_100m",
    "photoC_sp_zint_100m",
    "photoC_diaz_zint_100m",
]
lim_varnames_3d = [
    "TEMP",
    "SALT",
    "Fe",
    "NO3",
    "PO4",
    "SiO3",
    "UVEL",
    "VVEL",
    "Q",
]

lim_varnames_3d_w = [
    "WVEL",
    "QSW_3D",
]

lim_varnames_3d_150m = [
    "photoC_TOT",
    "diatChl",
    "diazChl",
    "spChl", 
    "diatC",
    "diazC",
    "spC",
    "diaz_Nfix",
    "photoFe_diat",
    "photoFe_diaz",
    "photoFe_sp",
    "photoNO3_diat",
    "photoNO3_diaz",
    "photoNO3_sp",
    "PO4_diat_uptake",
    "PO4_diaz_uptake",
    "PO4_sp_uptake",
    "PAR_avg",
    "zooC",
    "bSi_form",
]

cfgs = {}

for varname in lim_varnames_2d:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d_w:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d_150m:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')



In [5]:
# lim_varnames_2d = [
#     "TAUX",
#     "TAUY",
#     "sp_Fe_lim_surf",
#     "sp_N_lim_surf",
#     "sp_P_lim_surf",
#     "diat_Fe_lim_surf",
#     "diat_N_lim_surf",
#     "diat_P_lim_surf",
#     "diat_SiO3_lim_surf",
#     "HBLT",
#     "SSH",
#     "BSF",   
#     "LWDN_F",
#     "graze_diat_zoo_zint_100m",
#     "graze_diat_zint_100m",
#     "zoo_loss_zint_100m",
#     "IFRAC",
#     "sp_light_lim_surf",
#     "diat_light_lim_surf",
#     "diat_agg_zint_100m",
#     "sp_agg_zint_100m",
#     "photoC_diat_zint_100m",
#     "photoC_sp_zint_100m",
#     "photoC_diaz_zint_100m",
# ]

# for varname in lim_varnames_2d:
#     cfg = CESM2_config()
#     cfg.year_s = 1998
#     cfg.year_e = 2007
#     cfg.setvar(varname)
#     cfgs[varname] = cfg
    

In [6]:
# define preprocessing function

# ds_grid_sub = ds_grid.where(mask, drop=True)
# area = ds_grid_sub['TAREA']
# weighted_avg_manual = xr.Dataset()

exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
] + [cfg.var for cfg in cfgs.values()]

def process_coords_2d(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_t=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d_w(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_w_top=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d_150m(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_t_150m=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)


start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)

In [7]:
def get_preprocess_func(varname, sd, ed):
    if varname in lim_varnames_2d:
        return lambda ds: process_coords_2d(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d:
        return lambda ds: process_coords_3d(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d_w:
        return lambda ds: process_coords_3d_w(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d_150m:
        return lambda ds: process_coords_3d_150m(
            varname, ds, sd, ed
        )

    else:
        raise ValueError(f"[ERROR] Unknown varname: {varname}")

In [8]:
# Read A_F_REF1 dataset


import datetime
import numpy as np
import xarray as xr

start_time = time.time()

for varname, cfg in cfgs.items():
    
    # variable name
    dname = cfg.var

    preprocess_func = get_preprocess_func(
        dname, start_date, end_date
    )
    
# --------------------------------------------------
# load A_F_REF1 file paths
# --------------------------------------------------
    cfg.A_F_REF1_path_load(dname) 

    cfg.A_F_REF1_ds = xr.open_mfdataset(
        cfg.A_F_REF1_file_list[0],
        combine="nested",
        concat_dim=[[*cfg.A_F_REF1_ensembles], "time"],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.isel(concat_dim=0)
    new_time = cfg.A_F_REF1_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_REF1_ds.sizes["time"]
    )
    cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.assign_coords(time=new_time)
    # cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.compute()

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_REF1, ' + dname + ': ' + str(elapsed_time))


#------------------------------------------------------------------------------------------
    # load A_F_REF2 file paths
    cfg.A_F_REF2_path_load(dname)

    cfg.A_F_REF2_ds = xr.open_mfdataset(
        cfg.A_F_REF2_file_list[0],
        combine='nested',
        concat_dim=[[*cfg.A_F_REF2_ensembles], 'time'],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.isel(concat_dim=0)
    new_time = cfg.A_F_REF2_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_REF2_ds.sizes["time"]
    )
    cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.assign_coords(time=new_time)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_REF2, ' + dname + ': ' + str(elapsed_time))


#------------------------------------------------------------------------------------------
    # load A_F_WIND file paths
    cfg.A_F_WIND_path_load(dname)

    cfg.A_F_WIND_ds = xr.open_mfdataset(
        cfg.A_F_WIND_file_list[0],
        combine='nested',
        concat_dim=[[*cfg.A_F_WIND_ensembles], 'time'],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.isel(concat_dim=0)
    new_time = cfg.A_F_WIND_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_WIND_ds.sizes["time"]
    )
    cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.assign_coords(time=new_time)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_WIND, ' + dname + ': ' + str(elapsed_time))







elasped time for reading A_F_REF1, TAUX: 1.7618625164031982
elasped time for reading A_F_REF2, TAUX: 4.2062036991119385
elasped time for reading A_F_WIND, TAUX: 6.407668590545654
elasped time for reading A_F_REF1, TAUY: 8.428184032440186
elasped time for reading A_F_REF2, TAUY: 11.358738422393799
elasped time for reading A_F_WIND, TAUY: 15.397231340408325
elasped time for reading A_F_REF1, sp_Fe_lim_surf: 19.89416790008545
elasped time for reading A_F_REF2, sp_Fe_lim_surf: 24.08428120613098
elasped time for reading A_F_WIND, sp_Fe_lim_surf: 25.595750093460083
elasped time for reading A_F_REF1, sp_N_lim_surf: 29.794065952301025
elasped time for reading A_F_REF2, sp_N_lim_surf: 34.348421573638916
elasped time for reading A_F_WIND, sp_N_lim_surf: 37.918660402297974
elasped time for reading A_F_REF1, sp_P_lim_surf: 42.600231647491455
elasped time for reading A_F_REF2, sp_P_lim_surf: 46.136515617370605
elasped time for reading A_F_WIND, sp_P_lim_surf: 49.92092680931091
elasped time for read

In [9]:
# ## Variable reading for selected ones

# SELECT_VARS = [
#     "TAUX",
#     "TAUY",
#     "sp_Fe_lim_surf",
#     "sp_N_lim_surf",
#     "sp_P_lim_surf",
#     "diat_Fe_lim_surf",
#     "diat_N_lim_surf",
#     "diat_P_lim_surf",
#     "diat_SiO3_lim_surf",
#     "HBLT",
#     "SSH",
#     "BSF",   
#     "LWDN_F",
#     "graze_diat_zoo_zint_100m",
#     "graze_diat_zint_100m",
#     "zoo_loss_zint_100m",
#     "IFRAC",
#     "sp_light_lim_surf",
#     "diat_light_lim_surf",
#     "diat_agg_zint_100m",
#     "sp_agg_zint_100m",
#     "photoC_diat_zint_100m",
#     "photoC_sp_zint_100m",
#     "photoC_diaz_zint_100m",
# ]

# start_time = time.time()

# for varname in SELECT_VARS:

#     cfg = cfgs[varname]
#     dname = cfg.var

#     preprocess_func = get_preprocess_func(
#         dname, start_date, end_date
#     )

#     # --------------------------------------------------
#     # A_F_REF1
#     # --------------------------------------------------
#     cfg.A_F_REF1_path_load(dname)

#     cfg.A_F_REF1_ds = xr.open_mfdataset(
#         cfg.A_F_REF1_file_list[0],
#         combine="nested",
#         concat_dim=[[*cfg.A_F_REF1_ensembles], "time"],
#         preprocess=preprocess_func,
#         parallel=True,
#     )

#     cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.isel(concat_dim=0)

#     new_time = cfg.A_F_REF1_ds.time - np.array(
#         [datetime.timedelta(days=15)] * cfg.A_F_REF1_ds.sizes["time"]
#     )
#     cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.assign_coords(time=new_time)

#     print(f"[READ] A_F_REF1 {dname}")

#     # --------------------------------------------------
#     # A_F_REF2
#     # --------------------------------------------------
#     cfg.A_F_REF2_path_load(dname)

#     cfg.A_F_REF2_ds = xr.open_mfdataset(
#         cfg.A_F_REF2_file_list[0],
#         combine="nested",
#         concat_dim=[[*cfg.A_F_REF2_ensembles], "time"],
#         preprocess=preprocess_func,
#         parallel=True,
#     )

#     cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.isel(concat_dim=0)

#     new_time = cfg.A_F_REF2_ds.time - np.array(
#         [datetime.timedelta(days=15)] * cfg.A_F_REF2_ds.sizes["time"]
#     )
#     cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.assign_coords(time=new_time)

#     print(f"[READ] A_F_REF2 {dname}")

#     # --------------------------------------------------
#     # A_F_WIND
#     # --------------------------------------------------
#     cfg.A_F_WIND_path_load(dname)

#     cfg.A_F_WIND_ds = xr.open_mfdataset(
#         cfg.A_F_WIND_file_list[0],
#         combine="nested",
#         concat_dim=[[*cfg.A_F_WIND_ensembles], "time"],
#         preprocess=preprocess_func,
#         parallel=True,
#     )

#     cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.isel(concat_dim=0)

#     new_time = cfg.A_F_WIND_ds.time - np.array(
#         [datetime.timedelta(days=15)] * cfg.A_F_WIND_ds.sizes["time"]
#     )
#     cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.assign_coords(time=new_time)

#     print(f"[READ] A_F_WIND {dname}")

#     elapsed = time.time() - start_time
#     print(f"elapsed time for {dname}: {elapsed:.1f} s")


In [93]:

import gsw
import xarray as xr
import numpy as np

def compute_N2(temp, salt, zdim="z_t", g=9.81):
    """
    Compute Brunt–Väisälä frequency squared (N^2) using TEOS-10 (gsw).

    TEMP : degC (in-situ or potential, OK)
    SALT : PSU (practical salinity)
    zdim : vertical dimension (positive downward, meters)
    """

    # depth (positive down)
    z = temp[zdim]

    # pressure from depth (needs latitude; use grid mean)
    lat = temp["lat"].mean()
    p = xr.apply_ufunc(
        gsw.p_from_z,
        -z, lat,
        dask="parallelized",
        output_dtypes=[float],
    )

    # Absolute Salinity
    SA = xr.apply_ufunc(
        gsw.SA_from_SP,
        salt, p, temp["lon"], temp["lat"],
        dask="parallelized",
        output_dtypes=[float],
    )

    # Conservative Temperature
    CT = xr.apply_ufunc(
        gsw.CT_from_t,
        SA, temp, p,
        dask="parallelized",
        output_dtypes=[float],
    )

    # density
    rho = xr.apply_ufunc(
        gsw.rho,
        SA, CT, p,
        dask="parallelized",
        output_dtypes=[float],
    )

    # vertical gradient
    drho_dz = rho.differentiate(zdim)

    # N^2 = - g / rho * dρ/dz
    N2 = -g / rho * drho_dz

    N2.attrs["long_name"] = "Brunt–Väisälä frequency squared"
    N2.attrs["units"] = "s^-2"

    return N2



MODEL_KEYS = [
    "A_F_WIND_ds",
    "A_F_REF1_ds",
    "A_F_REF2_ds",
]


# ============================================================
# Add N2 to cfgs
# ============================================================
# cfgs["N2"] = {}

# ============================================================
# Create N2 config (same as other variables)
# ============================================================
cfg_N2 = CESM2_config()
cfg_N2.year_s = cfgs["TEMP"].year_s
cfg_N2.year_e = cfgs["TEMP"].year_e

cfgs["N2"] = cfg_N2

for model_key in MODEL_KEYS:

    if not hasattr(cfgs["TEMP"], model_key):
        continue
    if not hasattr(cfgs["SALT"], model_key):
        continue

    da_temp = getattr(cfgs["TEMP"], model_key)
    da_salt = getattr(cfgs["SALT"], model_key)

    if da_temp is None or da_salt is None:
        continue

    N2 = compute_N2(
        da_temp,
        da_salt,
        zdim="z_t"
    )

    setattr(cfgs["N2"], model_key, N2)

print("[DONE] N2 added as CESM2_config")



[DONE] N2 added as CESM2_config


In [ ]:
import numpy as np
import xarray as xr
import time

# ============================================================
# 0. Settings
# ============================================================

# (lon, lat) points to extract
# points = [
#     (-40, 20),
#     (-37, 50),
#     (100, -50),
#     (160, 10),
# ]
# points = [
#     (-40, 20),
#     (100, -50),
#     (-37, 50),
#     (160, 10),
# ]

points = [
    (100, -50),
    (-120, -55),
    (-37, -55),
    (35, -55),
]

# cfg attribute names for different model datasets
MODEL_KEYS = [
    "A_F_WIND_ds",
    "A_F_REF1_ds",
    "A_F_REF2_ds",
]

# ============================================================
# 1. Helper: extract and compute point data
# ============================================================

def extract_points_da(da, points):
    """
    Extract nearest grid point values for given lon/lat points
    and compute immediately.

    Returns:
        dict: {(lon,lat): DataArray}
    """
    out = {}
    for lon, lat in points:
        out[(lon, lat)] = (
            da
            .sel(lon=lon, lat=lat, method="nearest")
            .compute()
        )
    return out

# ============================================================
# 2. Main: loop over cfgs × models × points
# ============================================================

point_data = {}

start_time = time.time()

for varname, cfg in cfgs.items():

    point_data[varname] = {}

    for model_key in MODEL_KEYS:

        # skip if dataset does not exist
        if not hasattr(cfg, model_key):
            continue

        da = getattr(cfg, model_key)

        if da is None:
            continue

        # safety check
        if not all(dim in da.dims for dim in ["lat", "lon"]):
            print(f"[SKIP] {varname} {model_key} (no lat/lon)")
            continue

        # extract & compute
        point_data[varname][model_key] = extract_points_da(
            da, points
        )

    print(f"[DONE] {varname}")
    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for ' + f"[DONE] {varname}" + ': ' + str(elapsed_time))

# ============================================================
# 3. Usage examples
# ============================================================

# TEMP, REF1, (-40, 40)
# da_example = point_data["TEMP"]["A_F_REF1_ds"][(-40, 40)]
# print(da_example)

# # TEMP time series at two points
# p1 = point_data["TEMP"]["A_F_REF1_ds"][(-40, 40)]
# p2 = point_data["TEMP"]["A_F_REF1_ds"][(-40, 20)]


In [ ]:
import numpy as np
import xarray as xr
import time

# ============================================================
# 0. Settings
# ============================================================

# (lon, lat) center points
# CENTER_POINTS = [
#     (-40, 20),
#     (-37, 50),
# ]

# CENTER_POINTS = [
#     (-40, 20),
#     (100, -50),
#     (-37, 50),
#     (160, 10),
# ]
CENTER_POINTS = points

# model dataset attribute names
MODEL_KEYS = [
    "A_F_WIND_ds",
    "A_F_REF1_ds",
    "A_F_REF2_ds",
]

BOX_RADIUS = 2   # 2 → 5x5 box

# ============================================================
# 1. Helper: extract 5x5 box around a point
# ============================================================

def extract_box_da(da, lon0, lat0, radius=2):
    """
    Extract (2*radius+1) x (2*radius+1) box around (lon0, lat0).
    Keeps all vertical dimensions intact.

    Returns:
        DataArray with dims (..., lat, lon)
    """
    # find nearest indices
    ilon = int(np.abs(da.lon - lon0).argmin())
    ilat = int(np.abs(da.lat - lat0).argmin())

    lon_slice = slice(max(0, ilon - radius), ilon + radius + 1)
    lat_slice = slice(max(0, ilat - radius), ilat + radius + 1)

    box = da.isel(lon=lon_slice, lat=lat_slice)

    return box.compute()

# ============================================================
# 2. Main: loop over variables × models × points
# ============================================================

box_data = {}

start_time = time.time()

for varname, cfg in cfgs.items():

    box_data[varname] = {}

    for model_key in MODEL_KEYS:

        if not hasattr(cfg, model_key):
            continue

        da = getattr(cfg, model_key)
        if da is None:
            continue

        if not all(dim in da.dims for dim in ["lat", "lon"]):
            print(f"[SKIP] {varname} {model_key} (no lat/lon)")
            continue

        box_data[varname][model_key] = {}

        for lon0, lat0 in CENTER_POINTS:
            box_data[varname][model_key][(lon0, lat0)] = extract_box_da(
                da, lon0, lat0, radius=BOX_RADIUS
            )

    print(f"[DONE] {varname}")
    print(f"Elapsed time: {time.time() - start_time:.1f} s")

# ============================================================
# 3. Usage examples
# ============================================================

# Example: TEMP, REF1, 5x5 box around (-40, 40)
# da_box = box_data["TEMP"]["A_F_REF1_ds"][(-40, 40)]
# print(da_box)
# print(da_box.dims)

# Example: vertical profile still intact
# da_box.isel(lat=2, lon=2).plot()


In [ ]:
# container for WSM (wind stress magnitude)
box_data["WSM"] = {}

for mkey in box_data["TAUX"]:
    box_data["WSM"][mkey] = {}

    for pt in box_data["TAUX"][mkey]:
        taux = box_data["TAUX"][mkey][pt]
        tauy = box_data["TAUY"][mkey][pt]

        box_data["WSM"][mkey][pt] = np.sqrt(taux**2 + tauy**2)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np



# ============================================================
# 1. Vertical selector
# ============================================================

def select_surface(da, varname):
    if varname in lim_varnames_2d:
        return da
    elif varname in lim_varnames_3d:
        return da.isel(z_t=0)
    elif varname in lim_varnames_3d_150m:
        return da.isel(z_t_150m=0)
    elif varname in lim_varnames_3d_w:
        return da.isel(z_w_top=0)
    else:
        raise ValueError(f"Unknown variable type: {varname}")

# ============================================================
# 2. Plot settings
# ============================================================

# model_key = "A_F_REF1_ds"     # or A_F_WIND_ds / A_F_REF2_ds
model_keys = [
    "A_F_REF1_ds",
    "A_F_REF2_ds",
    "A_F_WIND_ds",
]


# point_labels = [
#     "(-40°, 20°)",
#     "(100°, -50°)",
#     "(-37°, 50°)",
#     "(160°, 10°)",
# ]
point_labels = [
    f"({lon}°, {lat}°)"
    for lon, lat in points
]


varnames = list(point_data.keys())
n_var = len(varnames)
n_pt  = len(points)

# ============================================================
# 3. Create figure
# ============================================================

fig, axes = plt.subplots(
    n_var, n_pt,
    figsize=(4 * n_pt, 1.8 * n_var),
    sharex=True,
    constrained_layout=True
)

# ============================================================
# 4. Main plotting loop
# ============================================================

model_labels = {
    "A_F_REF1_ds": "REF1",
    "A_F_REF2_ds": "REF2",
    "A_F_WIND_ds": "WIND",
}

for i, varname in enumerate(varnames):

    for j, point in enumerate(points):

        ax = axes[i, j] if n_var > 1 else axes[j]

        for k, model_key in enumerate(model_keys):

            if model_key not in point_data[varname]:
                continue

            da = point_data[varname][model_key][point]
            da = select_surface(da, varname)

            # cftime -> datetime64
            time_np = da.indexes["time"].to_datetimeindex()

            ax.plot(
                time_np,
                da.values,
                lw=1,
                label=model_labels[model_key]
            )

            # -------------------------------
            # correlation with photoC_TOT
            # -------------------------------
            if varname != "photoC_TOT":
    
                da_ref = point_data["photoC_TOT"][model_key][point]
                da_ref = select_surface(da_ref, "photoC_TOT")
    
                da1, da2 = xr.align(da, da_ref, join="inner")
    
                valid = np.isfinite(da1.values) & np.isfinite(da2.values)
    
                if valid.sum() > 10:
                    corr = np.corrcoef(
                        da1.values[valid],
                        da2.values[valid]
                    )[0, 1]
                else:
                    corr = np.nan
    
                ax.text(
                    0.02,
                    0.90 - 0.12 * k,
                    f"{model_labels[model_key]} r={corr:.2f}",
                    transform=ax.transAxes,
                    fontsize=7,
                    ha="left",
                    va="top"
                )
        
        # column titles (points)
        if i == 0:
            ax.set_title(point_labels[j], fontsize=10)
            ax.legend(fontsize=8, loc="upper right", frameon=False)

        # row labels (variables)
        if j == 0:
            ax.set_ylabel(varname, fontsize=9)

        ax.tick_params(labelsize=7)



# ============================================================
# 5. Final touches
# ============================================================

for ax in axes[-1, :]:
    ax.set_xlabel("Time", fontsize=9)

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# 1. Vertical selector
# ============================================================

def select_surface(da, varname):
    if varname in lim_varnames_2d:
        return da
    elif varname in lim_varnames_3d:
        return da.isel(z_t=0)
    elif varname in lim_varnames_3d_150m:
        return da.isel(z_t_150m=0)
    elif varname in lim_varnames_3d_w:
        return da.isel(z_w_top=0)
    else:
        raise ValueError(f"Unknown variable type: {varname}")

# ============================================================
# 2. Seasonal cycle removal
# ============================================================

def remove_seasonal_cycle(da):
    clim = da.groupby("time.month").mean("time")
    return da.groupby("time.month") - clim

# ============================================================
# 3. Preprocess time series (surface + anomaly)
# ============================================================

def preprocess_ts(da, varname):
    da = select_surface(da, varname)
    da = remove_seasonal_cycle(da)
    return da

# ============================================================
# 4. Plot settings
# ============================================================

model_keys = [
    "A_F_REF1_ds",
    "A_F_REF2_ds",
    "A_F_WIND_ds",
]

model_labels = {
    "A_F_REF1_ds": "REF1",
    "A_F_REF2_ds": "REF2",
    "A_F_WIND_ds": "WIND",
}

# points_2 = [(-40, 20), (-37, 50)]



varnames = list(point_data.keys())
n_var = len(varnames)
n_pt  = len(points)

# ============================================================
# 5. Create figure
# ============================================================

fig, axes = plt.subplots(
    n_var, n_pt,
    figsize=(4 * n_pt, 1.8 * n_var),
    sharex=True,
    constrained_layout=True
)

# ============================================================
# 6. Main plotting + correlation loop
# ============================================================

for i, varname in enumerate(varnames):

    for j, point in enumerate(points):

        ax = axes[i, j] if n_var > 1 else axes[j]

        for model_key in model_keys:

            if model_key not in point_data[varname]:
                continue
            if model_key not in point_data["photoC_TOT"]:
                continue

            # -------------------------------
            # variable anomaly
            # -------------------------------
            da = point_data[varname][model_key][point]
            da = preprocess_ts(da, varname)

            # cftime -> datetime64
            time_np = da.indexes["time"].to_datetimeindex()

            ax.plot(
                time_np,
                da.values,
                lw=1,
                label=model_labels[model_key]
            )

            # -------------------------------
            # correlation with photoC_TOT
            # -------------------------------
            if varname != "photoC_TOT":

                da_ref = point_data["photoC_TOT"][model_key][point]
                da_ref = preprocess_ts(da_ref, "photoC_TOT")

                da1, da2 = xr.align(da, da_ref, join="inner")

                valid = np.isfinite(da1.values) & np.isfinite(da2.values)

                if valid.sum() > 10:
                    corr = np.corrcoef(
                        da1.values[valid],
                        da2.values[valid]
                    )[0, 1]
                else:
                    corr = np.nan

                ax.text(
                    0.02, 0.85 - 0.12 * model_keys.index(model_key),
                    f"{model_labels[model_key]} r={corr:.2f}",
                    transform=ax.transAxes,
                    fontsize=7
                )

        # column titles (points)
        if i == 0:
            ax.set_title(point_labels[j], fontsize=10)
            ax.legend(fontsize=8, loc="upper right", frameon=False)

        # row labels (variables)
        if j == 0:
            ax.set_ylabel(varname, fontsize=9)

        ax.tick_params(labelsize=7)

# ============================================================
# 7. Final touches
# ============================================================

for ax in axes[-1, :]:
    ax.set_xlabel("Time", fontsize=9)

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# 1. Vertical selector
# ============================================================
def select_surface(da, varname):
    if varname in lim_varnames_2d:
        return da
    elif varname in lim_varnames_3d:
        return da.isel(z_t=0)
    elif varname in lim_varnames_3d_150m:
        return da.isel(z_t_150m=0)
    elif varname in lim_varnames_3d_w:
        return da.isel(z_w_top=0)
    else:
        raise ValueError(f"Unknown variable type: {varname}")

# ============================================================
# 2. Seasonal cycle removal
# ============================================================
def remove_seasonal_cycle(da):
    clim = da.groupby("time.month").mean("time")
    return da.groupby("time.month") - clim

# ============================================================
# 3. Preprocess time series
# ============================================================
def preprocess_ts(da, varname):
    da = select_surface(da, varname)
    da = remove_seasonal_cycle(da)
    return da

# ============================================================
# 4. User selection
# ============================================================
# 🔴 select variables
SELECT_VARS = [
    "photoC_TOT",
    "TAUX",
    "TAUY",
    "HBLT",
    "spC",
    "diatC",
    "UVEL",
    "VVEL",
    "TEMP",
    "SSH",
    "Q",
    "QSW_3D",
    "PAR_avg",
    "Fe",
    "NO3",
    "SiO3",
    "sp_N_lim_surf",
    "sp_Fe_lim_surf",
    "sp_light_lim_surf",
    "diat_Fe_lim_surf",
    "dian_N_lim_surf",
    "diat_light_lim_surf",
    "zooC",
]

model_keys = [
    "A_F_REF1_ds",
    "A_F_REF2_ds",
    "A_F_WIND_ds",
]

model_labels = {
    "A_F_REF1_ds": "REF1",
    "A_F_REF2_ds": "REF2",
    "A_F_WIND_ds": "WIND",
}

# ============================================================
# 5. Filter available variables
# ============================================================
varnames = [v for v in SELECT_VARS if v in point_data]
n_var = len(varnames)
n_pt  = len(points)

# ============================================================
# 6. Create figure
# ============================================================
fig, axes = plt.subplots(
    n_var, n_pt,
    figsize=(4 * n_pt, 1.8 * n_var),
    sharex=True,
    constrained_layout=True
)

if n_var == 1:
    axes = axes[np.newaxis, :]

# ============================================================
# 7. Main loop
# ============================================================
for i, varname in enumerate(varnames):

    for j, point in enumerate(points):

        ax = axes[i, j]

        for model_key in model_keys:

            if model_key not in point_data[varname]:
                continue
            if model_key not in point_data["photoC_TOT"]:
                continue

            # ---- variable anomaly
            da = point_data[varname][model_key][point]
            da = preprocess_ts(da, varname)

            time_np = da.indexes["time"].to_datetimeindex()

            ax.plot(
                time_np,
                da.values,
                lw=1,
                label=model_labels[model_key]
            )

            # ---- correlation with PP
            if varname != "photoC_TOT":

                da_ref = point_data["photoC_TOT"][model_key][point]
                da_ref = preprocess_ts(da_ref, "photoC_TOT")

                da1, da2 = xr.align(da, da_ref, join="inner")
                valid = np.isfinite(da1.values) & np.isfinite(da2.values)

                if valid.sum() > 10:
                    corr = np.corrcoef(
                        da1.values[valid],
                        da2.values[valid]
                    )[0, 1]
                else:
                    corr = np.nan

                ax.text(
                    0.02, 0.85 - 0.12 * model_keys.index(model_key),
                    f"{model_labels[model_key]} r={corr:.2f}",
                    transform=ax.transAxes,
                    fontsize=7
                )

        # ---- titles & labels
        if i == 0:
            ax.set_title(point_labels[j], fontsize=10)
            ax.legend(fontsize=8, loc="upper right", frameon=False)

        if j == 0:
            ax.set_ylabel(varname, fontsize=9)

        ax.tick_params(labelsize=7)

# ============================================================
# 8. X-labels
# ============================================================
for ax in axes[-1, :]:
    ax.set_xlabel("Time", fontsize=9)

plt.show()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ==========================================================
# Helper
# ==========================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

# ==========================================================
# Load data (surface)
# ==========================================================
da_q    = select_surface(cfgs["Q"].A_F_REF1_ds)
da_hblt = select_surface(cfgs["HBLT"].A_F_REF1_ds)
da_fe = select_surface(cfgs["QSW_3D"].A_F_REF1_ds)

# ==========================================================
# Time mean
# ==========================================================
q_mean    = da_q.mean("time")
hblt_mean = da_hblt.mean("time")
fe_mean = da_fe.mean("time")

lon = q_mean["lon"]
lat = q_mean["lat"]

# ==========================================================
# Plot: 2-panel map
# ==========================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=1,
    figsize=(8, 7),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

# ----------------------------------------------------------
# Panel 1: Q
# ----------------------------------------------------------
pcm0 = axes[0].pcolormesh(
    lon, lat, q_mean,
    transform=ccrs.PlateCarree(),
    shading="auto",
    cmap="viridis",
    vmin=-5e-9, vmax=0
)
axes[0].coastlines(linewidth=0.7)
axes[0].set_global()
axes[0].set_title("Surface Q (time mean)")

cbar0 = fig.colorbar(pcm0, ax=axes[0], shrink=0.85)
cbar0.set_label("Q")

# ----------------------------------------------------------
# Panel 2: HBLT
# ----------------------------------------------------------
pcm1 = axes[1].pcolormesh(
    lon, lat, hblt_mean,
    transform=ccrs.PlateCarree(),
    shading="auto",
    cmap="cividis",
    vmin=5000, vmax=12500
)
axes[1].coastlines(linewidth=0.7)
axes[1].set_global()
axes[1].set_title("HBLT (time mean)")

cbar1 = fig.colorbar(pcm1, ax=axes[1], shrink=0.85)
cbar1.set_label("HBLT (m)")

# ----------------------------------------------------------
# Panel 3: QSW_3D
# ----------------------------------------------------------
pcm1 = axes[2].pcolormesh(
    lon, lat, fe_mean,
    transform=ccrs.PlateCarree(),
    shading="auto",
    cmap="cividis",
    vmin=0, vmax=150,
)
axes[2].coastlines(linewidth=0.7)
axes[2].set_global()
axes[2].set_title("QSW_3D (time mean)")

cbar2 = fig.colorbar(pcm1, ax=axes[2], shrink=0.85)
cbar2.set_label("Fe")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ==========================================================
# Utilities
# ==========================================================
def movavg_12(da):
    """12-month centered moving average with edge trimming."""
    ma = da.rolling(time=12, center=True, min_periods=12).mean()
    return ma.isel(time=slice(6, -6))


def select_surface(da):
    """Automatically select surface if vertical dimension exists."""
    for zdim in ["z_t_150m", "z_t", "z_w_top"]:
        if zdim in da.dims:
            return da.isel({zdim: 0})
    return da


def select_box_center(da_box):
    """Select center grid point from a (2r+1)x(2r+1) box."""
    ilat = da_box.sizes["lat"] // 2
    ilon = da_box.sizes["lon"] // 2
    return da_box.isel(lat=ilat, lon=ilon)


# ==========================================================
# Settings
# ==========================================================
VARS = ["diatC", "spC", "diazC"]
MODEL_KEY = "A_F_REF1_ds"

POINTS = CENTER_POINTS          # already defined
# POINT_LABELS = [f"P{i+1}" for i in range(len(POINTS))]
POINT_LABELS = [
    f"P{i+1} ({lon}°, {lat}°)"
    for i, (lon, lat) in enumerate(POINTS)
]
COLORS = {
    "diatC": "tab:blue",
    "spC": "tab:orange",
    "diazC": "tab:green",
}


# ==========================================================
# 1. Extract REF1 center-point time series (12M MA)
# ==========================================================
ts_data = []

for pt in POINTS:

    ts_vars = {}

    for var in VARS:
        da_box = box_data[var][MODEL_KEY][pt]

        da_ctr = select_box_center(da_box)
        da_ctr = select_surface(da_ctr)
        da_ma  = movavg_12(da_ctr)

        ts_vars[var] = da_ma

    ts_data.append(ts_vars)

# ==========================================================
# 2. Plot: 2 panels (one per point)
# ==========================================================
fig, axes = plt.subplots(
    nrows=1, ncols=2,
    figsize=(18, 6),
    sharex=True,
    constrained_layout=True
)

for i, ax in enumerate(axes):

    time_np = ts_data[i][VARS[0]].indexes["time"].to_datetimeindex()

    for var in VARS:
        ax.plot(
            time_np,
            ts_data[i][var],
            label=var,
            lw=2,
            color=COLORS[var]
        )

    ax.set_title(f"REF1 – 12-month MA (Center point {POINT_LABELS[i]})")
    
    ax.set_ylabel("Carbon concentration")
    ax.legend(loc="upper left", fontsize=9)

axes[-1].set_xlabel("Time")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ==========================================================
# Utilities (same as before)
# ==========================================================
def movavg_12(da):
    ma = da.rolling(time=12, center=True, min_periods=12).mean()
    return ma.isel(time=slice(6, -6))


def select_surface(da):
    for zdim in ["z_t_150m", "z_t", "z_w_top"]:
        if zdim in da.dims:
            return da.isel({zdim: 0})
    return da


def select_box_center(da_box):
    ilat = da_box.sizes["lat"] // 2
    ilon = da_box.sizes["lon"] // 2
    return da_box.isel(lat=ilat, lon=ilon)


# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
POINTS = CENTER_POINTS
POINT_LABELS = [
    f"P{i+1} ({lon}°, {lat}°)"
    for i, (lon, lat) in enumerate(POINTS)
]

# ---- variables -------------------------------------------------
SP_VARS = [
    "sp_Fe_lim_surf",
    "sp_N_lim_surf",
    "sp_P_lim_surf",
]

DIAT_VARS = [
    "diat_Fe_lim_surf",
    "diat_N_lim_surf",
    "diat_P_lim_surf",
    "diat_SiO3_lim_surf",
]

# ---- nutrient → color mapping ---------------------------------
NUTRIENT_COLORS = {
    "Fe": "tab:red",
    "N": "tab:blue",
    "P": "tab:green",
    "SiO3": "tab:orange",
}

# ---- species → linestyle --------------------------------------
LINESTYLE = {
    "sp": "-",
    "diat": "--",
}


def get_nutrient(varname):
    """Extract nutrient name from variable string."""
    for nut in ["Fe", "N", "P", "SiO3"]:
        if f"_{nut}_" in varname:
            return nut
    raise ValueError(f"Unknown nutrient in {varname}")


def get_species(varname):
    return "sp" if varname.startswith("sp_") else "diat"


# ==========================================================
# 1. Extract time series
# ==========================================================
ts_data = []

for pt in POINTS:

    ts_vars = {}

    for var in SP_VARS + DIAT_VARS:
        da_box = box_data[var][MODEL_KEY][pt]
        da_ctr = select_surface(select_box_center(da_box))
        ts_vars[var] = da_ctr

    ts_data.append(ts_vars)


# ==========================================================
# 2. Plot: 2 columns × N points
# ==========================================================
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(20, 6),
    sharex=True,
    constrained_layout=True
)

# ==========================================================
# Column 1: sp only
# ==========================================================
ax = axes[0]

time_np = ts_data[0][SP_VARS[0]].indexes["time"].to_datetimeindex()

for var in SP_VARS:
    nut = get_nutrient(var)
    ax.plot(
        time_np,
        ts_data[0][var],
        color=NUTRIENT_COLORS[nut],
        linestyle=LINESTYLE["sp"],
        lw=2,
        label=f"sp – {nut}"
    )

ax.set_title(f"REF1 – sp nutrient limitation {POINT_LABELS[0]}")
ax.set_ylabel("Limitation factor")
ax.legend(fontsize=9)


# ==========================================================
# Column 2: sp + diat
# ==========================================================
ax = axes[1]

for var in SP_VARS + DIAT_VARS:
    nut = get_nutrient(var)
    spc = get_species(var)

    ax.plot(
        time_np,
        ts_data[1][var],
        color=NUTRIENT_COLORS[nut],
        linestyle=LINESTYLE[spc],
        lw=2,
        label=f"{spc} – {nut}"
    )

ax.set_title(f"REF1 – sp & diat nutrient limitation {POINT_LABELS[1]}")
ax.legend(fontsize=9, ncol=2)
axes[-1].set_xlabel("Time")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ==========================================================
# Utilities (same as before)
# ==========================================================
def movavg_12(da):
    ma = da.rolling(time=12, center=True, min_periods=12).mean()
    return ma.isel(time=slice(6, -6))


def select_surface(da):
    for zdim in ["z_t_150m", "z_t", "z_w_top"]:
        if zdim in da.dims:
            return da.isel({zdim: 0})
    return da


def select_box_center(da_box):
    ilat = da_box.sizes["lat"] // 2
    ilon = da_box.sizes["lon"] // 2
    return da_box.isel(lat=ilat, lon=ilon)


# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
POINTS = CENTER_POINTS

# ---- variables -------------------------------------------------
SP_VARS = [
    "sp_Fe_lim_surf",
    "sp_N_lim_surf",
    "sp_P_lim_surf",
]

DIAT_VARS = [
    "diat_Fe_lim_surf",
    "diat_N_lim_surf",
    "diat_P_lim_surf",
    "diat_SiO3_lim_surf",
]

# ---- nutrient → color mapping ---------------------------------
NUTRIENT_COLORS = {
    "Fe": "tab:red",
    "N": "tab:blue",
    "P": "tab:green",
    "SiO3": "tab:orange",
}

# ---- species → linestyle --------------------------------------
LINESTYLE = {
    "sp": "-",
    "diat": "--",
}


def get_nutrient(varname):
    """Extract nutrient name from variable string."""
    for nut in ["Fe", "N", "P", "SiO3"]:
        if f"_{nut}_" in varname:
            return nut
    raise ValueError(f"Unknown nutrient in {varname}")


def get_species(varname):
    return "sp" if varname.startswith("sp_") else "diat"


# ==========================================================
# 1. Extract time series
# ==========================================================
ts_data = []

for pt in POINTS:

    ts_vars = {}

    for var in SP_VARS + DIAT_VARS:
        da_box = box_data[var][MODEL_KEY][pt]
        da_ctr = select_surface(select_box_center(da_box))
        ts_vars[var] = movavg_12(da_ctr)

    ts_data.append(ts_vars)


# ==========================================================
# 2. Plot: 2 columns × N points
# ==========================================================
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(20, 6),
    sharex=True,
    constrained_layout=True
)

# ==========================================================
# Column 1: sp only
# ==========================================================
ax = axes[0]

time_np = ts_data[0][SP_VARS[0]].indexes["time"].to_datetimeindex()

for var in SP_VARS:
    nut = get_nutrient(var)
    ax.plot(
        time_np,
        ts_data[0][var],
        color=NUTRIENT_COLORS[nut],
        linestyle=LINESTYLE["sp"],
        lw=2,
        label=f"sp – {nut}"
    )

ax.set_title(f"REF1 – sp nutrient limitation (12M MA) {POINT_LABELS[0]}")

ax.set_ylabel("Limitation factor")
ax.legend(fontsize=9)


# ==========================================================
# Column 2: sp + diat
# ==========================================================
ax = axes[1]

for var in SP_VARS + DIAT_VARS:
    nut = get_nutrient(var)
    spc = get_species(var)

    ax.plot(
        time_np,
        ts_data[1][var],
        color=NUTRIENT_COLORS[nut],
        linestyle=LINESTYLE[spc],
        lw=2,
        label=f"{spc} – {nut}"
    )

ax.set_title(f"REF1 – sp & diat nutrient limitation (12M MA) {POINT_LABELS[1]}")
ax.legend(fontsize=9, ncol=2)

axes[-1].set_xlabel("Time")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Lead–lag correlation utility
# ==========================================================
def lead_lag_corr(x, y, max_lag=24):
    """
    Corr between x(t) and y(t+lag)
    lag > 0 : y leads x
    lag < 0 : x leads y
    """
    lags = np.arange(-max_lag, max_lag + 1)
    r = np.full(len(lags), np.nan)

    for i, lag in enumerate(lags):
        if lag < 0:
            xx = x[:lag]
            yy = y[-lag:]
        elif lag > 0:
            xx = x[lag:]
            yy = y[:-lag]
        else:
            xx = x
            yy = y

        valid = np.isfinite(xx) & np.isfinite(yy)
        if valid.sum() > 10:
            r[i] = np.corrcoef(xx[valid], yy[valid])[0, 1]

    return lags, r

PO4_VAR  = "PO4"
SIO3_VAR = "SiO3"
TEMP_VAR  = "TEMP"
PAR_VAR = "PAR_AVG"
DIAT_VAR = "diatC"
SP_VAR = "spC"

ts_data = []

for pt in POINTS:

    ts = {}

    for var in [PHOTO_VAR, NO3_VAR, FE_VAR, SIO3_VAR, PO4_VAR, TEMP_VAR, DIAT_VAR, SP_VAR, SIO3_VAR, "HBLT", "TAUX"]:
        da_box = box_data[var][MODEL_KEY][pt]
        da_ctr = select_surface(select_box_center(da_box))
        da_anom = remove_seasonal_mean(da_ctr)

        ts[var] = zscore(da_anom)

    ts_data.append(ts)

# ==========================================================
# Settings
# ==========================================================
MAX_LAG = 24

PANEL_CFG = {
    0: [NO3_VAR, PHOTO_VAR, SP_VAR],            # Panel 1
    1: ["Fe", "HBLT", "TAUX", "spC"],   # Panel 2
}

COLORS = {
    "NO3": "tab:blue",
    "HBLT": "tab:blue",
    "spC": "tab:olive",
    "Fe": "tab:orange",
    "photoC_TOT": "tab:green",
    "SiO3": "tab:red",
    "TAUX": "tab:red",
}


# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(18, 5),
    sharey=True,
    constrained_layout=True
)

for ip, ax in enumerate(axes):

    photo = ts_data[ip]["photoC_TOT"].values

    for nut in PANEL_CFG[ip]:
        lags, r = lead_lag_corr(
            photo,
            ts_data[ip][nut].values,
            max_lag=MAX_LAG
        )

        # ---- line plot ----
        ax.plot(
            lags, r,
            lw=2.5,
            color=COLORS[nut],
            label=nut
        )

        # ---- find max |r| lag ----
        imax = np.nanargmax(np.abs(r))
        lag_max = lags[imax]
        r_max = r[imax]

        # ---- direction text ----
        if lag_max > 0:
            direction = f"{nut} → PP"
            x_text = lag_max + 1
        elif lag_max < 0:
            direction = f"PP → {nut}"
            x_text = lag_max - 1
        else:
            direction = f"PP = {nut}"
            x_text = lag_max + 1


        # ---- arrow + text ----
        ax.annotate(
            direction,
            xy=(lag_max, r_max),
            xytext=(x_text, r_max),
            arrowprops=dict(
                arrowstyle="->",
                color=COLORS[nut],
                lw=1.5
            ),
            color=COLORS[nut],
            fontsize=10,
            va="center"
        )

    # ---- reference lines ----
    ax.axvline(0, color="k", lw=1)
    ax.axhline(0, color="k", lw=0.8, alpha=0.5)

    ax.set_xlim(-MAX_LAG, MAX_LAG)
    ax.set_xlabel("Lag (months)")
    ax.set_xticks(np.arange(-MAX_LAG, MAX_LAG + 1, 2))
    ax.set_title(f"{POINT_LABELS[ip]}: PP ↔ others lead–lag")
    ax.legend(loc="best")

axes[0].set_ylabel("Correlation")

plt.show()


In [ ]:
# ==========================================================
# Containers (ip별 저장)
# ==========================================================
ADV_X_list = []
ADV_Y_list = []
ADV_Z_list = []

UVEL_VAR = "UVEL"
VVEL_VAR = "VVEL"
WVEL_VAR = "WVEL"

# ==========================================================
# Loop over points
# ==========================================================
for ip, pt in enumerate(POINTS):

    Fe_box   = box_data[FE_VAR][MODEL_KEY][pt]
    U_box    = box_data[UVEL_VAR][MODEL_KEY][pt]
    V_box    = box_data[VVEL_VAR][MODEL_KEY][pt]
    W_box    = box_data[WVEL_VAR][MODEL_KEY][pt]

    ilat = Fe_box.sizes["lat"] // 2
    ilon = Fe_box.sizes["lon"] // 2

    Fe_surf  = select_surface(Fe_box)
    Fe_below = Fe_box.isel(z_t=1)

    U_surf = select_surface(U_box)
    V_surf = select_surface(V_box)
    W_lev1 = W_box.isel(z_w_top=1)

    # X
    Fe_east = Fe_surf.isel(lat=ilat, lon=ilon + 1)
    Fe_west = Fe_surf.isel(lat=ilat, lon=ilon - 1)
    U_ctr   = U_surf.isel(lat=ilat, lon=ilon)

    ADV_X = U_ctr * (Fe_east - Fe_west)

    # Y
    Fe_north = Fe_surf.isel(lat=ilat + 1, lon=ilon)
    Fe_south = Fe_surf.isel(lat=ilat - 1, lon=ilon)
    V_ctr    = V_surf.isel(lat=ilat, lon=ilon)

    ADV_Y = V_ctr * (Fe_north - Fe_south)

    # Z
    Fe_ctr_surf  = Fe_surf.isel(lat=ilat, lon=ilon)
    Fe_ctr_below = Fe_below.isel(lat=ilat, lon=ilon)
    W_ctr        = W_lev1.isel(lat=ilat, lon=ilon)

    ADV_Z = W_ctr * (Fe_ctr_surf - Fe_ctr_below)

    # ======================================================
    # Store (중요!!)
    # ======================================================
    ADV_X_list.append(ADV_X)
    ADV_Y_list.append(ADV_Y)
    ADV_Z_list.append(ADV_Z)


# ==========================================================
# Utility
# ==========================================================
def corr_da(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return np.nan
    return np.corrcoef(x[valid], y[valid])[0, 1]


# ==========================================================
# Correlation: PP vs advection terms
# ==========================================================
for ip, pt in enumerate(POINTS):

    print(f"\nPoint {POINT_LABELS[ip]}")

    # --------------------------------
    # PP (already in memory)
    # --------------------------------
    PP = ts_data[ip][PHOTO_VAR]   # seasonal anomaly / zscore whatever 네가 쓰던 그대로

    # --------------------------------
    # Advection terms (already computed above)
    # 여기서는 네가 계산해 둔 변수를 그대로 씀
    # --------------------------------
    ADV_X = ADV_X_list[ip]   # or ADV_X_dict[pt]
    ADV_Y = ADV_Y_list[ip]
    ADV_Z = ADV_Z_list[ip]

    # --------------------------------
    # Correlations
    # --------------------------------
    r_x = corr_da(PP.values, ADV_X.values)
    r_y = corr_da(PP.values, ADV_Y.values)
    r_z = corr_da(PP.values, ADV_Z.values)

    print(f"  corr(PP, ADV_X) = {r_x:.2f}")
    print(f"  corr(PP, ADV_Y) = {r_y:.2f}")
    print(f"  corr(PP, ADV_Z) = {r_z:.2f}")


In [ ]:
import numpy as np

# ==========================================================
# Utility
# ==========================================================
def corr_da(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return np.nan
    return np.corrcoef(x[valid], y[valid])[0, 1]


# ==========================================================
# Nutrients to process
# ==========================================================
NUTRIENT_VARS = {
    "Fe":   "Fe",
    "SiO3": "SiO3",
    "NO3":  "NO3",
}

UVEL_VAR = "UVEL"
VVEL_VAR = "VVEL"
WVEL_VAR = "WVEL"


# ==========================================================
# Containers: nutrient → ADV lists
# ==========================================================
ADV_X = {nut: [] for nut in NUTRIENT_VARS}
ADV_Y = {nut: [] for nut in NUTRIENT_VARS}
ADV_Z = {nut: [] for nut in NUTRIENT_VARS}


# ==========================================================
# 1. Compute advection for each nutrient & point
# ==========================================================
for ip, pt in enumerate(POINTS):

    # velocities (same for all nutrients)
    U_box = box_data[UVEL_VAR][MODEL_KEY][pt]
    V_box = box_data[VVEL_VAR][MODEL_KEY][pt]
    W_box = box_data[WVEL_VAR][MODEL_KEY][pt]

    U_surf = select_surface(U_box)
    V_surf = select_surface(V_box)
    W_lev1 = W_box.isel(z_w_top=1)

    ilat = U_surf.sizes["lat"] // 2
    ilon = U_surf.sizes["lon"] // 2

    U_ctr = U_surf.isel(lat=ilat, lon=ilon)
    V_ctr = V_surf.isel(lat=ilat, lon=ilon)
    W_ctr = W_lev1.isel(lat=ilat, lon=ilon)

    # loop over nutrients
    for nut_name, nut_var in NUTRIENT_VARS.items():

        N_box = box_data[nut_var][MODEL_KEY][pt]
        N_surf = select_surface(N_box)
        N_below = N_box.isel(z_t=1)

        # X
        N_east = N_surf.isel(lat=ilat, lon=ilon + 1)
        N_west = N_surf.isel(lat=ilat, lon=ilon - 1)
        adv_x = U_ctr * (N_east - N_west)

        # Y
        N_north = N_surf.isel(lat=ilat + 1, lon=ilon)
        N_south = N_surf.isel(lat=ilat - 1, lon=ilon)
        adv_y = V_ctr * (N_north - N_south)

        # Z
        N_ctr_surf  = N_surf.isel(lat=ilat, lon=ilon)
        N_ctr_below = N_below.isel(lat=ilat, lon=ilon)
        adv_z = W_ctr * (N_ctr_surf - N_ctr_below)

        # store
        ADV_X[nut_name].append(adv_x)
        ADV_Y[nut_name].append(adv_y)
        ADV_Z[nut_name].append(adv_z)


# ==========================================================
# 2. Correlation: PP vs advection (for all nutrients)
# ==========================================================
for ip, pt in enumerate(POINTS):

    PP = ts_data[ip][PHOTO_VAR]

    print(f"\nPoint {POINT_LABELS[ip]}")

    for nut in NUTRIENT_VARS.keys():

        r_x = corr_da(PP.values, ADV_X[nut][ip].values)
        r_y = corr_da(PP.values, ADV_Y[nut][ip].values)
        r_z = corr_da(PP.values, ADV_Z[nut][ip].values)

        print(f"  [{nut}] corr(PP, ADV_X) = {r_x:.2f}")
        print(f"  [{nut}] corr(PP, ADV_Y) = {r_y:.2f}")
        print(f"  [{nut}] corr(PP, ADV_Z) = {r_z:.2f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def select_vertical(da, nlev):
    """
    Select upper nlev levels regardless of vertical dimension name.
    """
    if "z_t" in da.dims:
        return da.isel(z_t=slice(0, nlev)), "z_t"
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=slice(0, nlev)), "z_t_150m"
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=slice(0, nlev)), "z_w_top"
    else:
        raise ValueError(f"No known vertical dimension in {da.dims}")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
NLEV = 15
CMAP = "viridis"

# p2 only
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# Variables
HOV_VARS = {
    "Fe": "Fe",
    "SiO3": "SiO3",
    "PAR_avg": "PAR_avg",
    "QSW_3D":  "QSW_3D",
    "zooC":    "zooC",
    "Q": "Q",
}

TS_VARS = {
    "PP": "photoC_TOT",
    "Diatom light limitation": "diat_light_lim_surf",
    "Downward LW radiation":   "LWDN_F",
    "Diatom grazing":          "graze_diat_zint_100m",
    "Diatom aggregation":      "diat_agg_zint_100m",
}

# ==========================================================
# Time
# ==========================================================
time_np = (
    select_surface(
        select_box_center(box_data["photoC_TOT"][MODEL_KEY][pt])
    )
    .indexes["time"]
    .to_datetimeindex()
)

# ==========================================================
# Figure
# ==========================================================
nrows = len(HOV_VARS) + len(TS_VARS)
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=1,
    figsize=(16, 3 * nrows),
    sharex=True,
    constrained_layout=False
)

row = 0
pcm_list = []

# ==========================================================
# Time-series panels
# ==========================================================
for name, var in TS_VARS.items():

    ax = axes[row]
    da = box_data[var][MODEL_KEY][pt]
    ctr = select_surface(select_box_center(da))

    ax.plot(time_np, ctr.values, color="k", lw=1.8)
    ax.set_ylabel(name)
    ax.set_title(name)

    row += 1

axes[-1].set_xlabel("Time")
fig.suptitle(f"{pt_label}: light, forcing & ecosystem diagnostics", y=0.90)

# ==========================================================
# Hovmöller panels
# ==========================================================
for name, var in HOV_VARS.items():

    ax = axes[row]
    da = box_data[var][MODEL_KEY][pt]
    ctr0 = select_box_center(da)
    ctr, zdim = select_vertical(ctr0, NLEV)
    
    depth = ctr[zdim].values[:NLEV]
    
    pcm = ax.pcolormesh(
        time_np,
        depth,
        ctr.transpose(zdim, "time").values,
        shading="auto",
        cmap=CMAP
    )

    pcm_list.append(pcm)

    ax.invert_yaxis()
    ax.set_ylabel(f"{name}\nDepth (m)")
    ax.set_title(name)

    row += 1



# ==========================================================
# Single colorbar for Hovmöller
# ==========================================================
cbar = fig.colorbar(
    pcm_list[-1],
    ax=axes[:len(HOV_VARS)+len(TS_VARS)],
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)
cbar.set_label("Value")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def select_vertical(da, nlev):
    """
    Select upper nlev levels regardless of vertical dimension name.
    """
    if "z_t" in da.dims:
        return da.isel(z_t=slice(0, nlev)), "z_t"
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=slice(0, nlev)), "z_t_150m"
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=slice(0, nlev)), "z_w_top"
    else:
        raise ValueError(f"No known vertical dimension in {da.dims}")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
NLEV = 15
CMAP = "viridis"

# p2 only
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# Variables
HOV_VARS = {
    "SiO3": "SiO3",
    "zooC":    "zooC",
    "spC": "spC",
}

TS_VARS = {
    "PP": "photoC_TOT",
    "Diatom grazing":          "graze_diat_zint_100m",
    "HBLT": "HBLT",
}

# ==========================================================
# Time
# ==========================================================
time_np = (
    select_surface(
        select_box_center(box_data["photoC_TOT"][MODEL_KEY][pt])
    )
    .indexes["time"]
    .to_datetimeindex()
)

# ==========================================================
# Figure
# ==========================================================
nrows = len(HOV_VARS) + len(TS_VARS)
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=1,
    figsize=(16, 3 * nrows),
    sharex=True,
    constrained_layout=False
)

row = 0
pcm_list = []

# ==========================================================
# Time-series panels
# ==========================================================
for name, var in TS_VARS.items():

    ax = axes[row]
    da = box_data[var][MODEL_KEY][pt]
    ctr = select_surface(select_box_center(da))

    ax.plot(time_np, ctr.values, color="k", lw=1.8)
    ax.set_ylabel(name)
    ax.set_title(name)

    row += 1

axes[-1].set_xlabel("Time")
fig.suptitle(f"{pt_label}: light, forcing & ecosystem diagnostics", y=0.90)

# ==========================================================
# Hovmöller panels
# ==========================================================
for name, var in HOV_VARS.items():

    ax = axes[row]
    da = box_data[var][MODEL_KEY][pt]
    ctr0 = select_box_center(da)
    ctr, zdim = select_vertical(ctr0, NLEV)
    
    depth = ctr[zdim].values[:NLEV]
    
    pcm = ax.pcolormesh(
        time_np,
        depth,
        ctr.transpose(zdim, "time").values,
        shading="auto",
        cmap=CMAP
    )

    pcm_list.append(pcm)

    ax.invert_yaxis()
    ax.set_ylabel(f"{name}\nDepth (m)")
    ax.set_title(name)

    row += 1



# ==========================================================
# Single colorbar for Hovmöller
# ==========================================================
cbar = fig.colorbar(
    pcm_list[-1],
    ax=axes[:len(HOV_VARS)+len(TS_VARS)],
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)
cbar.set_label("Value")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Lead–lag correlation utility
# ==========================================================
def lead_lag_corr(x, y, max_lag=24):
    """
    Corr between x(t) and y(t+lag)
    lag > 0 : y leads x
    lag < 0 : x leads y
    """
    lags = np.arange(-max_lag, max_lag + 1)
    r = np.full(len(lags), np.nan)

    for i, lag in enumerate(lags):
        if lag < 0:
            xx = x[:lag]
            yy = y[-lag:]
        elif lag > 0:
            xx = x[lag:]
            yy = y[:-lag]
        else:
            xx = x
            yy = y

        valid = np.isfinite(xx) & np.isfinite(yy)
        if valid.sum() > 10:
            r[i] = np.corrcoef(xx[valid], yy[valid])[0, 1]

    return lags, r

SIO3_VAR = "SiO3"
PO4_VAR  = "PO4"
TEMP_VAR  = "TEMP"
PAR_VAR = "PAR_AVG"
DIAT_VAR = "diatC"
SP_VAR = "spC"
DIAT_GRAZ_VAR = "graze_diat_zint_100m"
HBLT_VAR = "HBLT"
zooC_VAR = "zooC"


ts_data = []

for pt in POINTS:

    ts = {}

    for var in [PHOTO_VAR, NO3_VAR, SIO3_VAR, TEMP_VAR, DIAT_VAR, SP_VAR, HBLT_VAR, DIAT_GRAZ_VAR, zooC_VAR]:
        da_box = box_data[var][MODEL_KEY][pt]
        da_ctr = select_surface(select_box_center(da_box))
        da_anom = remove_seasonal_mean(da_ctr)

        ts[var] = zscore(da_anom)

    ts_data.append(ts)

# ==========================================================
# Settings
# ==========================================================
MAX_LAG = 24

PANEL_CFG = {
    0: [TEMP_VAR, NO3_VAR, SP_VAR],            # Panel 1
    1: [ DIAT_VAR, DIAT_GRAZ_VAR, HBLT_VAR, SIO3_VAR, zooC_VAR],   # Panel 2
}

COLORS = {
    NO3_VAR: "tab:blue",
    SIO3_VAR: "tab:blue",
    PO4_VAR: "tab:red",
    HBLT_VAR: "tab:red",
    TEMP_VAR: "tab:green",
    DIAT_VAR: "tab:orange",
    SP_VAR: "tab:olive",
    DIAT_GRAZ_VAR: "tab:olive",
    zooC_VAR: "tab:green",
}


# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(18, 5),
    sharey=True,
    constrained_layout=True
)

for ip, ax in enumerate(axes):

    photo = ts_data[ip][PHOTO_VAR].values

    for nut in PANEL_CFG[ip]:
        lags, r = lead_lag_corr(
            photo,
            ts_data[ip][nut].values,
            max_lag=MAX_LAG
        )

        # ---- line plot ----
        ax.plot(
            lags, r,
            lw=2.5,
            color=COLORS[nut],
            label=nut
        )

        # ---- find max |r| lag ----
        imax = np.nanargmax(np.abs(r))
        lag_max = lags[imax]
        r_max = r[imax]

        # ---- direction text ----
        if lag_max > 0:
            direction = f"{nut} → PP"
            x_text = lag_max + 1
        elif lag_max < 0:
            direction = f"PP → {nut}"
            x_text = lag_max - 1
        else:
            direction = f"PP = {nut}"
            x_text = lag_max + 1


        # ---- arrow + text ----
        ax.annotate(
            direction,
            xy=(lag_max, r_max),
            xytext=(x_text, r_max),
            arrowprops=dict(
                arrowstyle="->",
                color=COLORS[nut],
                lw=1.5
            ),
            color=COLORS[nut],
            fontsize=10,
            va="center"
        )

    # ---- reference lines ----
    ax.axvline(0, color="k", lw=1)
    ax.axhline(0, color="k", lw=0.8, alpha=0.5)

    ax.set_xlim(-MAX_LAG, MAX_LAG)
    ax.set_xlabel("Lag (months)")
    ax.set_xticks(np.arange(-MAX_LAG, MAX_LAG + 1, 2))
    ax.set_title(f"{POINT_LABELS[ip]}: PP ↔ nutrient lead–lag")
    ax.legend(loc="best")

axes[0].set_ylabel("Correlation")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Helper: vertical selector
# ==========================================================
def select_vertical(da, nlev):
    if "z_t" in da.dims:
        return da.isel(z_t=slice(0, nlev)), "z_t"
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=slice(0, nlev)), "z_t_150m"
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=slice(0, nlev)), "z_w_top"
    else:
        raise ValueError(f"No known vertical dimension in {da.dims}")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
NLEV = 15
CMAP = "viridis"

# p2 only
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# Variables
HOV_VARS = {
    "SiO3": "SiO3",
    "PAR_avg":  "PAR_avg",
    "QSW_3D":  "QSW_3D",
    
}

TS_VARS = {
    "PP": "photoC_TOT",
    "Diatom light limitation": "diat_light_lim_surf",
    "TAUX": "TAUX",
    "TAUY": "TAUY",
    "WSM": "WSM", 
    "HBLT": "HBLT",
}

# ==========================================================
# Time
# ==========================================================
time_np = (
    select_surface(
        select_box_center(box_data["photoC_TOT"][MODEL_KEY][pt])
    )
    .indexes["time"]
    .to_datetimeindex()
)

# ==========================================================
# Pre-compute wind stress magnitude |tau|
# ==========================================================
da_tx = box_data["TAUX"][MODEL_KEY][pt]
da_ty = box_data["TAUY"][MODEL_KEY][pt]

tx = select_surface(select_box_center(da_tx)).values
ty = select_surface(select_box_center(da_ty)).values

tau_mag = np.sqrt(tx**2 + ty**2)

# ==========================================================
# Figure
# ==========================================================
nrows = len(HOV_VARS) + len(TS_VARS)
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=1,
    figsize=(16, 3 * nrows),
    sharex=True,
    constrained_layout=False
)

row = 0
pcm_list = []

# ==========================================================
# Time-series panels
# ==========================================================
for name, var in TS_VARS.items():

    ax = axes[row]

    if name == "Wind stress magnitude":
        ax.plot(time_np, tau_mag, color="tab:blue", lw=1.8)
        ax.set_ylabel("|τ| (N m$^{-2}$)")
    else:
        da = box_data[var][MODEL_KEY][pt]
        ctr = select_surface(select_box_center(da))
        ax.plot(time_np, ctr.values, color="k", lw=1.8)
        ax.set_ylabel(name)

    ax.set_title(name)
    row += 1

axes[-1].set_xlabel("Time")
fig.suptitle(f"{pt_label}: light, forcing & ecosystem diagnostics", y=0.90)

# ==========================================================
# Hovmöller panels
# ==========================================================
for name, var in HOV_VARS.items():

    ax = axes[row]
    da = box_data[var][MODEL_KEY][pt]
    ctr0 = select_box_center(da)
    ctr, zdim = select_vertical(ctr0, NLEV)

    depth = ctr[zdim].values[:NLEV]

    pcm = ax.pcolormesh(
        time_np,
        depth,
        ctr.transpose(zdim, "time").values,
        shading="auto",
        cmap=CMAP
    )

    pcm_list.append(pcm)

    ax.invert_yaxis()
    ax.set_ylabel(f"{name}\nDepth (m)")
    ax.set_title(name)

    row += 1

# ==========================================================
# Single colorbar for Hovmöller
# ==========================================================
cbar = fig.colorbar(
    pcm_list[-1],
    ax=axes[len(TS_VARS):],   # hovmöller block만
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)
cbar.set_label("Value")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Helper: vertical selector
# ==========================================================
def select_vertical(da, nlev):
    if "z_t" in da.dims:
        return da.isel(z_t=slice(0, nlev)), "z_t"
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=slice(0, nlev)), "z_t_150m"
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=slice(0, nlev)), "z_w_top"
    else:
        raise ValueError(f"No known vertical dimension in {da.dims}")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
NLEV = 15
CMAP = "viridis"

# p2 only
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# ==========================================================
# Variables
# ==========================================================
# --- Hovmöller (vertical structure only) ---
HOV_VARS = {
    "Fe": "Fe",
}

# --- Time series (surface) ---
TS_VARS = {
    "PP": "photoC_TOT",
    "Diatom light limitation": "diat_light_lim_surf",
    "PAR_avg (surface)": "PAR_avg",
    "QSW_3D (surface)": "QSW_3D",
    "WSM": "WSM",
    "HBLT": "HBLT",
}

# ==========================================================
# Time
# ==========================================================
time_np = (
    select_surface(
        select_box_center(box_data["photoC_TOT"][MODEL_KEY][pt])
    )
    .indexes["time"]
    .to_datetimeindex()
)

# ==========================================================
# Pre-compute wind stress magnitude |tau|
# ==========================================================
da_tx = box_data["TAUX"][MODEL_KEY][pt]
da_ty = box_data["TAUY"][MODEL_KEY][pt]

tx = select_surface(select_box_center(da_tx)).values
ty = select_surface(select_box_center(da_ty)).values

tau_mag = np.sqrt(tx**2 + ty**2)

# ==========================================================
# Figure
# ==========================================================
nrows = len(TS_VARS) + len(HOV_VARS)
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=1,
    figsize=(16, 3 * nrows),
    sharex=True,
    constrained_layout=False
)

row = 0
pcm_list = []

# ==========================================================
# Time-series panels (ALL surface)
# ==========================================================
for name, var in TS_VARS.items():

    ax = axes[row]

    if var in ["TAUX", "TAUY"]:
        da = box_data[var][MODEL_KEY][pt]
        ctr = select_surface(select_box_center(da))
        ax.plot(time_np, ctr.values, lw=1.8)
        ax.set_ylabel(name)

    elif name == "Wind stress magnitude":
        ax.plot(time_np, tau_mag, lw=1.8)
        ax.set_ylabel("|τ| (N m$^{-2}$)")

    else:
        da = box_data[var][MODEL_KEY][pt]
        ctr = select_surface(select_box_center(da))
        ax.plot(time_np, ctr.values, lw=1.8)
        ax.set_ylabel(name)

    ax.set_title(name)
    row += 1

# ==========================================================
# Hovmöller panels
# ==========================================================
for name, var in HOV_VARS.items():

    ax = axes[row]

    da = box_data[var][MODEL_KEY][pt]
    ctr0 = select_box_center(da)
    ctr, zdim = select_vertical(ctr0, NLEV)

    depth = ctr[zdim].values[:NLEV]

    pcm = ax.pcolormesh(
        time_np,
        depth,
        ctr.transpose(zdim, "time").values,
        shading="auto",
        cmap=CMAP
    )

    pcm_list.append(pcm)

    ax.invert_yaxis()
    ax.set_ylabel(f"{name}\nDepth (m)")
    ax.set_title(name)

    row += 1

# ==========================================================
# Colorbar (Hovmöller only)
# ==========================================================
cbar = fig.colorbar(
    pcm_list[-1],
    ax=axes[len(TS_VARS):],
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)
cbar.set_label("Value")

# ==========================================================
# Final touches
# ==========================================================
axes[-1].set_xlabel("Time")
fig.suptitle(
    f"{pt_label}: surface light/forcing & subsurface nutrient structure",
    y=0.92
)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# ==========================================================
# Helper: vertical selector
# ==========================================================
def select_vertical(da, nlev):
    if "z_t" in da.dims:
        return da.isel(z_t=slice(0, nlev)), "z_t"
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=slice(0, nlev)), "z_t_150m"
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=slice(0, nlev)), "z_w_top"
    else:
        raise ValueError(f"No known vertical dimension in {da.dims}")

def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

# ==========================================================
# Helper: deseasonalize (monthly climatology removal)
# ==========================================================
def deseasonalize(da):
    clim = da.groupby("time.month").mean("time", skipna=True)
    return da.groupby("time.month") - clim

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
NLEV = 15
CMAP = "viridis"

# p2 only
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# ==========================================================
# Variables
# ==========================================================
HOV_VARS = {
    "Fe": "Fe",
}

TS_VARS = {
    "PP": "photoC_TOT",
    "Diatom light limitation": "diat_light_lim_surf",
    "PAR_avg (surface)": "PAR_avg",
    "QSW_3D (surface)": "QSW_3D",
    "WSM": "WSM",
    "HBLT": "HBLT",
}

# ==========================================================
# Time
# ==========================================================
time_np = (
    deseasonalize(
        select_surface(
            select_box_center(box_data["photoC_TOT"][MODEL_KEY][pt])
        )
    )
    .indexes["time"]
    .to_datetimeindex()
)

# ==========================================================
# Pre-compute wind stress magnitude |tau| (deseasonalized)
# ==========================================================
da_tx = deseasonalize(
    select_surface(select_box_center(box_data["TAUX"][MODEL_KEY][pt]))
)
da_ty = deseasonalize(
    select_surface(select_box_center(box_data["TAUY"][MODEL_KEY][pt]))
)

tau_mag = np.sqrt(da_tx.values**2 + da_ty.values**2)

# ==========================================================
# Figure
# ==========================================================
nrows = len(TS_VARS) + len(HOV_VARS)
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=1,
    figsize=(16, 3 * nrows),
    sharex=True,
    constrained_layout=False
)

row = 0
pcm_list = []

# ==========================================================
# Time-series panels (surface, deseasonalized)
# ==========================================================
for name, var in TS_VARS.items():

    ax = axes[row]

    da = box_data[var][MODEL_KEY][pt]
    ctr = deseasonalize(
        select_surface(select_box_center(da))
    )

    ax.plot(time_np, ctr.values, lw=1.8)
    ax.set_ylabel(name)
    ax.set_title(name)

    row += 1

# ==========================================================
# Hovmöller panels (deseasonalized)
# ==========================================================
for name, var in HOV_VARS.items():

    ax = axes[row]

    da = box_data[var][MODEL_KEY][pt]
    ctr0 = deseasonalize(select_box_center(da))
    ctr, zdim = select_vertical(ctr0, NLEV)

    depth = ctr[zdim].values[:NLEV]

    pcm = ax.pcolormesh(
        time_np,
        depth,
        ctr.transpose(zdim, "time").values,
        shading="auto",
        cmap=CMAP
    )

    pcm_list.append(pcm)

    ax.invert_yaxis()
    ax.set_ylabel(f"{name}\nDepth (m)")
    ax.set_title(name)

    row += 1

# ==========================================================
# Colorbar (Hovmöller only)
# ==========================================================
cbar = fig.colorbar(
    pcm_list[-1],
    ax=axes[len(TS_VARS):],
    orientation="horizontal",
    fraction=0.04,
    pad=0.02
)
cbar.set_label("Anomaly")

# ==========================================================
# Final touches
# ==========================================================
axes[-1].set_xlabel("Time")
fig.suptitle(
    f"{pt_label}: deseasonalized surface forcing & subsurface structure",
    y=0.92
)

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr

def remove_seasonal_cycle_1d(da):
    clim = da.groupby("time.month").mean("time")
    return da.groupby("time.month") - clim

def remove_seasonal_cycle_2d(da):
    clim = da.groupby("time.month").mean("time")
    return da.groupby("time.month") - clim

# ==========================================================
# Build cross-correlation table (seasonal cycle removed)
# ==========================================================
ts_dict = {}

# ----------------------------------------------------------
# 1. Time-series variables (anomaly)
# ----------------------------------------------------------
for name, var in TS_VARS.items():
    da = box_data[var][MODEL_KEY][pt]
    ctr = select_surface(select_box_center(da))

    ctr_anom = remove_seasonal_cycle_1d(ctr)
    ts_dict[name] = ctr_anom.values

# ----------------------------------------------------------
# 2. Hovmöller variables → upper-ocean mean → anomaly
# ----------------------------------------------------------
for name, var in HOV_VARS.items():
    da = box_data[var][MODEL_KEY][pt]
    ctr0 = select_box_center(da)
    ctr, zdim = select_vertical(ctr0, NLEV)

    ts = ctr.mean(dim=zdim)
    ts_anom = remove_seasonal_cycle_1d(ts)

    ts_dict[name] = ts_anom.values

# ----------------------------------------------------------
# 3. Correlation table
# ----------------------------------------------------------
df_corr = pd.DataFrame(ts_dict, index=time_np)
corr_table = df_corr.corr().round(2)

# ==========================================================
# 4. Mask upper triangle
# ==========================================================
mask = np.triu(np.ones_like(corr_table, dtype=bool), k=1)
corr_masked = corr_table.mask(mask)

# ==========================================================
# 5. Discrete heatmap (0.1 bins)
# ==========================================================
bounds = np.arange(-1.0, 1.01, 0.1)
norm = mcolors.BoundaryNorm(bounds, ncolors=256)
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad(color="white")

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(
    corr_masked.values,
    cmap=cmap,
    norm=norm,
    aspect="auto"
)

# Axis labels
ax.set_xticks(np.arange(len(corr_table.columns)))
ax.set_yticks(np.arange(len(corr_table.index)))
ax.set_xticklabels(corr_table.columns, rotation=45, ha="right")
ax.set_yticklabels(corr_table.index)

# Grid
ax.set_xticks(np.arange(-0.5, len(corr_table.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(corr_table.index), 1), minor=True)
ax.grid(which="minor", color="gray", linestyle="-", linewidth=0.3)
ax.tick_params(which="minor", bottom=False, left=False)

ax.set_title("Cross-correlation between variables (seasonal cycle removed)")

# Colorbar
cbar = fig.colorbar(
    im,
    ax=ax,
    ticks=bounds,
    spacing="uniform"
)
cbar.set_label("Correlation")

plt.tight_layout()
plt.show()

# ==========================================================
# 6. Print table
# ==========================================================
print(corr_table)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Lead–lag correlation utility
# ==========================================================
def lead_lag_corr(x, y, max_lag=24):
    """
    Corr between x(t) and y(t+lag)
    lag > 0 : y leads x
    lag < 0 : x leads y
    """
    lags = np.arange(-max_lag, max_lag + 1)
    r = np.full(len(lags), np.nan)

    for i, lag in enumerate(lags):
        if lag < 0:
            xx = x[:lag]
            yy = y[-lag:]
        elif lag > 0:
            xx = x[lag:]
            yy = y[:-lag]
        else:
            xx = x
            yy = y

        valid = np.isfinite(xx) & np.isfinite(yy)
        if valid.sum() > 10:
            r[i] = np.corrcoef(xx[valid], yy[valid])[0, 1]

    return lags, r

SIO3_VAR = "SiO3"
PAR_VAR = "PAR_avg"
DIAT_VAR = "diatC"
QSW_3D_VAR = "QSW_3D"
HBLT_VAR = "HBLT"
WSM_VAR = "WSM"


ts_data = []

for pt in POINTS:

    ts = {}

    for var in [PHOTO_VAR, SIO3_VAR, DIAT_VAR, PAR_VAR, HBLT_VAR, QSW_3D_VAR, WSM_VAR, SP_VAR]:
        da_box = box_data[var][MODEL_KEY][pt]
        da_ctr = select_surface(select_box_center(da_box))
        da_anom = remove_seasonal_mean(da_ctr)

        ts[var] = zscore(da_anom)

    ts_data.append(ts)

# ==========================================================
# Settings
# ==========================================================
MAX_LAG = 24

PANEL_CFG = {
    0: [SIO3_VAR, SP_VAR, PAR_VAR, HBLT_VAR, QSW_3D_VAR, WSM_VAR],            # Panel 1
    1: [ SIO3_VAR, SP_VAR, PAR_VAR, HBLT_VAR, QSW_3D_VAR, WSM_VAR],   # Panel 2
}

COLORS = {
    NO3_VAR: "tab:blue",
    SIO3_VAR: "tab:blue",
    PO4_VAR: "tab:red",
    HBLT_VAR: "tab:red",
    TEMP_VAR: "tab:green",
    SP_VAR: "tab:orange",
    WSM_VAR: "tab:pink",
    PAR_VAR: "tab:olive",
    QSW_3D_VAR: "tab:green",
}


# ==========================================================
# Plot
# ==========================================================
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(18, 5),
    sharey=True,
    constrained_layout=True
)

for ip, ax in enumerate(axes):

    photo = ts_data[ip][PHOTO_VAR].values

    for nut in PANEL_CFG[ip]:
        lags, r = lead_lag_corr(
            photo,
            ts_data[ip][nut].values,
            max_lag=MAX_LAG
        )

        # ---- line plot ----
        ax.plot(
            lags, r,
            lw=2.5,
            color=COLORS[nut],
            label=nut
        )

        # ---- find max |r| lag ----
        imax = np.nanargmax(np.abs(r))
        lag_max = lags[imax]
        r_max = r[imax]

        # ---- direction text ----
        if lag_max > 0:
            direction = f"{nut} → PP"
            x_text = lag_max + 1
        elif lag_max < 0:
            direction = f"PP → {nut}"
            x_text = lag_max - 1
        else:
            direction = f"PP = {nut}"
            x_text = lag_max + 1


        # ---- arrow + text ----
        ax.annotate(
            direction,
            xy=(lag_max, r_max),
            xytext=(x_text, r_max),
            arrowprops=dict(
                arrowstyle="->",
                color=COLORS[nut],
                lw=1.5
            ),
            color=COLORS[nut],
            fontsize=10,
            va="center"
        )

    # ---- reference lines ----
    ax.axvline(0, color="k", lw=1)
    ax.axhline(0, color="k", lw=0.8, alpha=0.5)

    ax.set_xlim(-MAX_LAG, MAX_LAG)
    ax.set_xlabel("Lag (months)")
    ax.set_xticks(np.arange(-MAX_LAG, MAX_LAG + 1, 2))
    ax.set_title(f"{POINT_LABELS[ip]}: PP ↔ nutrient lead–lag")
    ax.legend(loc="best")

axes[0].set_ylabel("Correlation")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import calendar

# ==========================================================
# Helper: seasonal mean by year
# ==========================================================
def seasonal_mean(da, months):
    da_sel = da.where(da["time.month"].isin(months), drop=True)
    return da_sel.groupby("time.year").mean("time")

def month_label(months):
    # e.g. [2,3,4] -> "Feb–Apr"
    names = [calendar.month_abbr[m] for m in months]
    if len(names) == 1:
        return names[0]
    else:
        return f"{names[0]}–{names[-1]}"

# ==========================================================
# Settings (ONLY change here)
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
pt = POINTS[1]
pt_label = POINT_LABELS[1]

# forcing months → response months
FORCING_MONTHS  = [12]   # Feb–Apr
RESPONSE_MONTHS = [12]      # May–Jun

forcing_lab  = month_label(FORCING_MONTHS)
response_lab = month_label(RESPONSE_MONTHS)

# ==========================================================
# Variables to compare
# ==========================================================
FORCING_VARS = {
    "WSM":  "WSM",
    "TAUX": "TAUX",
    "TAUY": "TAUY",
    "HBLT": "HBLT",
}

# ==========================================================
# Response variable (PP)
# ==========================================================
da_pp = box_data["photoC_TOT"][MODEL_KEY][pt]
pp_ctr = select_surface(select_box_center(da_pp))
pp_seas = seasonal_mean(pp_ctr, RESPONSE_MONTHS)

# ==========================================================
# Figure
# ==========================================================
fig, axes = plt.subplots(2, 2, figsize=(9, 9), constrained_layout=True)
axes = axes.flatten()

for ax, (vname, key) in zip(axes, FORCING_VARS.items()):

    da_var = box_data[key][MODEL_KEY][pt]
    var_ctr = select_surface(select_box_center(da_var))
    var_seas = seasonal_mean(var_ctr, FORCING_MONTHS)

    # match common years
    common_years = np.intersect1d(var_seas["year"], pp_seas["year"])
    x = var_seas.sel(year=common_years).values
    y = pp_seas.sel(year=common_years).values

    # correlation
    r = np.corrcoef(x, y)[0, 1]

    # scatter
    ax.scatter(x, y, color="k", s=25)

    # regression line
    coef = np.polyfit(x, y, 1)
    xfit = np.linspace(x.min(), x.max(), 100)
    yfit = coef[0] * xfit + coef[1]
    ax.plot(xfit, yfit, color="tab:red", lw=1.5)

    ax.set_xlabel(f"{vname} ({forcing_lab})")
    ax.set_ylabel(f"PP ({response_lab})")
    ax.set_title(f"{vname} vs PP\nr = {r:.2f}")
    ax.grid(True)

fig.suptitle(f"{pt_label}: Forcing → PP relationship", fontsize=13)
plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ==========================================================
# Helper: monthly mean by year
# ==========================================================
def monthly_mean_by_year(da, month):
    da_sel = da.where(da["time.month"] == month, drop=True)
    return da_sel.groupby("time.year").mean("time")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
months = np.arange(1, 13)
POINTS = points

# ==========================================================
# Figure
# ==========================================================
fig, axes = plt.subplots(
    nrows=len(POINTS), ncols=1,
    figsize=(7, 3.5 * len(POINTS)),
    sharex=True,
    constrained_layout=True
)

if len(POINTS) == 1:
    axes = [axes]

# ==========================================================
# Loop over points
# ==========================================================
for ax, pt, pt_label in zip(axes, POINTS, POINT_LABELS):

    da_wsm = box_data["WSM"][MODEL_KEY][pt]
    da_pp  = box_data["photoC_TOT"][MODEL_KEY][pt]

    wsm_ctr = select_surface(select_box_center(da_wsm))
    pp_ctr  = select_surface(select_box_center(da_pp))

    r_month = []

    for m in months:
        wsm_m = monthly_mean_by_year(wsm_ctr, m)
        pp_m  = monthly_mean_by_year(pp_ctr, m)

        common_years = np.intersect1d(wsm_m["year"], pp_m["year"])
        wsm_m = wsm_m.sel(year=common_years)
        pp_m  = pp_m.sel(year=common_years)

        if len(common_years) > 2:
            r = np.corrcoef(wsm_m.values, pp_m.values)[0, 1]
        else:
            r = np.nan

        r_month.append(r)

    r_month = np.array(r_month)

    ax.plot(months, r_month, marker="o", lw=2, color="k")
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_ylabel("Correlation (r)")
    ax.set_title(pt_label)
    ax.grid(True, alpha=0.3)

# ==========================================================
# X-axis
# ==========================================================
axes[-1].set_xticks(months)
axes[-1].set_xticklabels(
    ["Jan","Feb","Mar","Apr","May","Jun",
     "Jul","Aug","Sep","Oct","Nov","Dec"]
)
axes[-1].set_xlabel("Month")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import calendar

# ==========================================================
# Helper
# ==========================================================
def monthly_mean_by_year(da, month):
    da_sel = da.where(da["time.month"] == month, drop=True)
    return da_sel.groupby("time.year").mean("time")

def months_label(months):
    names = [calendar.month_abbr[m] for m in months]
    if len(names) == 1:
        return names[0]
    else:
        return f"{names[0]}–{names[-1]}"

# ==========================================================
# SETTINGS (ONLY change here)
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
pt = POINTS[1]

FORCING_MONTHS  = np.arange(1, 13)   # months to loop over (default: all)
RESPONSE_MONTHS = [12]             # PP response months

resp_lab = months_label(RESPONSE_MONTHS)

# ==========================================================
# Load pt-based time series
# ==========================================================
# QSW_3D
da_qsw = box_data["QSW_3D"][MODEL_KEY][pt]
qsw_ctr = select_surface(select_box_center(da_qsw))

# HBLT
da_hblt = box_data["HBLT"][MODEL_KEY][pt]
hblt_ctr = select_surface(select_box_center(da_hblt))

# WSM
da_wsm = box_data["WSM"][MODEL_KEY][pt]
wsm_ctr = select_surface(select_box_center(da_wsm))

# PP (response)
da_pp = box_data["photoC_TOT"][MODEL_KEY][pt]
pp_ctr = select_surface(select_box_center(da_pp))
pp_resp = (
    pp_ctr
    .where(pp_ctr["time.month"].isin(RESPONSE_MONTHS), drop=True)
    .groupby("time.year")
    .mean("time")
)

# ==========================================================
# Monthly correlations
# ==========================================================
months = FORCING_MONTHS

r_qsw_pp   = []
r_hblt_pp  = []
r_wsm_pp   = []

r_hblt_qsw = []
r_wsm_qsw  = []
r_wsm_hblt = []

for m in months:
    qsw_m  = monthly_mean_by_year(qsw_ctr, m)
    hblt_m = monthly_mean_by_year(hblt_ctr, m)
    wsm_m  = monthly_mean_by_year(wsm_ctr, m)

    # match years
    common_years = np.intersect1d(
        qsw_m["year"],
        np.intersect1d(
            hblt_m["year"],
            np.intersect1d(wsm_m["year"], pp_resp["year"])
        )
    )

    qsw_m  = qsw_m.sel(year=common_years)
    hblt_m = hblt_m.sel(year=common_years)
    wsm_m  = wsm_m.sel(year=common_years)
    pp_m   = pp_resp.sel(year=common_years)

    # --- PP response
    r_qsw_pp.append(np.corrcoef(qsw_m.values, pp_m.values)[0, 1])
    r_hblt_pp.append(np.corrcoef(hblt_m.values, pp_m.values)[0, 1])
    r_wsm_pp.append(np.corrcoef(wsm_m.values, pp_m.values)[0, 1])

    # --- physical couplings
    r_hblt_qsw.append(np.corrcoef(hblt_m.values, qsw_m.values)[0, 1])
    r_wsm_qsw.append(np.corrcoef(wsm_m.values, qsw_m.values)[0, 1])
    r_wsm_hblt.append(np.corrcoef(wsm_m.values, hblt_m.values)[0, 1])

r_qsw_pp   = np.array(r_qsw_pp)
r_hblt_pp  = np.array(r_hblt_pp)
r_wsm_pp   = np.array(r_wsm_pp)

r_hblt_qsw = np.array(r_hblt_qsw)
r_wsm_qsw  = np.array(r_wsm_qsw)
r_wsm_hblt = np.array(r_wsm_hblt)

# ==========================================================
# Plot (2 panels: response / physics)
# ==========================================================
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

# --- (a) PP response
ax = axes[0]
ax.plot(months, r_qsw_pp,  marker="o", lw=2, label=f"QSW(m) ↔ PP({resp_lab})")
ax.plot(months, r_hblt_pp, marker="s", lw=2, label=f"HBLT(m) ↔ PP({resp_lab})")
ax.plot(months, r_wsm_pp,  marker="^", lw=2, label=f"WSM(m) ↔ PP({resp_lab})")
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_ylabel("Correlation")
ax.set_title(f"(a) Controls on PP ({resp_lab}) at {pt}")
ax.legend(ncol=2)
ax.grid(alpha=0.3)

# --- (b) Physical coupling
ax = axes[1]
ax.plot(months, r_hblt_qsw, marker="o", lw=2, label="HBLT(m) ↔ QSW(m)")
ax.plot(months, r_wsm_qsw,  marker="s", lw=2, label="WSM(m) ↔ QSW(m)")
ax.plot(months, r_wsm_hblt, marker="^", lw=2, label="WSM(m) ↔ HBLT(m)")
ax.axhline(0, color="gray", lw=1, ls="--")
ax.set_xlabel("Month")
ax.set_ylabel("Correlation")
ax.set_title(f"(b) Physical coupling at {pt}")
ax.legend(ncol=2)
ax.grid(alpha=0.3)

plt.xticks(months)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Helper
# ==========================================================
def monthly_mean_by_year(da, month):
    da_sel = da.where(da["time.month"] == month, drop=True)
    return da_sel.groupby("time.year").mean("time")

# ==========================================================
# Settings
# ==========================================================
MODEL_KEY = "A_F_REF1_ds"
pt = POINTS[1]

months = np.arange(1, 13)
lags = np.arange(-6, 7)   # -6 to +6 months

# ==========================================================
# Load pt-based time series
# ==========================================================
# HBLT
da_hblt = box_data["HBLT"][MODEL_KEY][pt]
hblt_ctr = select_surface(select_box_center(da_hblt))

# QSW
da_qsw = box_data["QSW_3D"][MODEL_KEY][pt]
qsw_ctr = select_surface(select_box_center(da_qsw))

# ==========================================================
# Lead–lag correlation matrix
# ==========================================================
R = np.full((len(months), len(lags)), np.nan)

for i, m in enumerate(months):
    hblt_m = monthly_mean_by_year(hblt_ctr, m)

    for j, lag in enumerate(lags):
        m_qsw = m + lag

        # month wrapping (1–12)
        if m_qsw < 1 or m_qsw > 12:
            continue

        qsw_m = monthly_mean_by_year(qsw_ctr, m_qsw)

        # match years
        common_years = np.intersect1d(hblt_m["year"], qsw_m["year"])
        if len(common_years) < 3:
            continue

        h = hblt_m.sel(year=common_years).values
        q = qsw_m.sel(year=common_years).values

        R[i, j] = np.corrcoef(h, q)[0, 1]

# ==========================================================
# Plot lead–lag correlation map
# ==========================================================
plt.figure(figsize=(10, 5))

im = plt.imshow(
    R,
    origin="lower",
    aspect="auto",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1
)

cbar = plt.colorbar(im)
cbar.set_label("Correlation")

# x / y ticks
plt.xticks(
    np.arange(len(lags)),
    lags
)
plt.yticks(
    np.arange(len(months)),
    ["Jan","Feb","Mar","Apr","May","Jun",
     "Jul","Aug","Sep","Oct","Nov","Dec"]
)

plt.xlabel("Lag (months)")
plt.ylabel("Center month of HBLT")

plt.title("Lead–lag correlation between HBLT and QSW")

# ----------------------------------------------------------
# Zero-lag reference line
# ----------------------------------------------------------
zero_idx = np.where(lags == 0)[0][0]
plt.axvline(zero_idx, color="k", lw=1)

# ----------------------------------------------------------
# Lead–lag direction annotations (IMPORTANT PART)
# ----------------------------------------------------------
ax = plt.gca()
ny, nx = R.shape

# QSW leads HBLT (lag < 0)
ax.annotate(
    "QSW leads HBLT",
    xy=(0.05, 1.02),
    xytext=(0.05, 1.02),
    xycoords="axes fraction",
    textcoords="axes fraction",
    # arrowprops=dict(arrowstyle="<-", lw=1.5),
    ha="center",
    va="bottom"
)

# HBLT leads QSW (lag > 0)
ax.annotate(
    "HBLT leads QSW",
    xy=(0.70, 1.02),
    xytext=(0.95, 1.02),
    xycoords="axes fraction",
    textcoords="axes fraction",
    # arrowprops=dict(arrowstyle="->", lw=1.5),
    ha="center",
    va="bottom"
)

# ----------------------------------------------------------
# Optional: label lag=0 explicitly
# ----------------------------------------------------------
ax.text(
    zero_idx, -0.6,
    "lag = 0",
    ha="center",
    va="top"
)

plt.tight_layout()
plt.show()



In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def remove_seasonal_cycle(da):
    return da.groupby("time.month") - da.groupby("time.month").mean("time")

# ============================================================
# Load PP (photoC_TOT) experiments
# ============================================================
da_ref  = cfgs["photoC_TOT"].A_F_REF1_ds
da_xall = cfgs["photoC_TOT"].A_F_REF2_ds
da_wind = cfgs["photoC_TOT"].A_F_WIND_ds

# surface only
da_ref  = select_surface(da_ref)
da_xall = select_surface(da_xall)
da_wind = select_surface(da_wind)

# ============================================================
# Remove seasonal cycle
# ============================================================
ref_anom  = remove_seasonal_cycle(da_ref)
xall_anom = remove_seasonal_cycle(da_xall)
wind_anom = remove_seasonal_cycle(da_wind)

# ============================================================
# Correlations
# ============================================================
corr_xALL = xr.corr(ref_anom, xall_anom, dim="time")
corr_WIND = xr.corr(ref_anom, wind_anom, dim="time")
corr_DIFF = corr_WIND - corr_xALL

# ============================================================
# Plot: 3 panels
# ============================================================
lon = corr_xALL["lon"]
lat = corr_xALL["lat"]

proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=1,
    figsize=(6, 8),
    constrained_layout=True,
    subplot_kw=dict(projection=proj),
)


pcm0 = axes[0].pcolormesh(
    lon, lat, corr_xALL,
    cmap="RdBu_r", vmin=-1, vmax=1, shading="auto", transform=ccrs.PlateCarree(),
)
axes[0].set_title("PP corr: xALL")

pcm1 = axes[1].pcolormesh(
    lon, lat, corr_WIND,
    cmap="RdBu_r", vmin=-1, vmax=1, shading="auto", transform=ccrs.PlateCarree(),
)
axes[1].set_title("PP corr: WIND")

pcm2 = axes[2].pcolormesh(
    lon, lat, corr_DIFF,
    cmap="RdBu_r", vmin=-1, vmax=1, shading="auto", transform=ccrs.PlateCarree(),
)
axes[2].set_title("corr(WIND) − corr(xALL)")

for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

cbar = fig.colorbar(pcm2, ax=axes, shrink=0.9)
cbar.set_label("Correlation")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ============================================================
# Settings
# ============================================================
THRESH = 0.5
SUBSAMPLE = 4
ALPHA = 0.35
MSIZE = 6

mask_sig = (corr_DIFF > THRESH)

# ============================================================
# Discrete colormap with minimal duplication (12 unique colors)
# ============================================================
MONTHS = np.arange(1, 13)
bounds = np.arange(0.5, 13.5, 1)

# option A: tab12 (matplotlib ≥3.7)
try:
    cmap = plt.get_cmap("tab12")
except ValueError:
    # option B: custom ListedColormap (guaranteed unique)
    cmap = mcolors.ListedColormap([
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
        "#bcbd22", "#17becf", "#aec7e8", "#ffbb78"
    ])

norm = mcolors.BoundaryNorm(bounds, cmap.N)

# ============================================================
# Prepare subsampled dot coordinates
# ============================================================
lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

mask_sub = np.zeros_like(mask_sig.values, dtype=bool)
mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True
mask_plot = mask_sig.values & mask_sub

sig_lon = LON[mask_plot]
sig_lat = LAT[mask_plot]

# ============================================================
# Plot with sparse dot overlay
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=1,
    figsize=(10, 12),
    constrained_layout=True,
    subplot_kw=dict(projection=proj)
)

pcm0 = axes[0].pcolormesh(
    lon, lat, pp_peak_month,
    transform=ccrs.PlateCarree(),
    cmap=cmap, norm=norm, shading="auto"
)
axes[0].scatter(
    sig_lon, sig_lat,
    transform=ccrs.PlateCarree(),
    s=MSIZE, c="k", alpha=ALPHA, marker="."
)
fig.colorbar(pcm0, ax=axes[0], ticks=MONTHS, label="Month")
axes[0].set_title("PP: Month of Maximum")
axes[0].coastlines()

pcm1 = axes[1].pcolormesh(
    lon, lat, wsm_min_month,
    transform=ccrs.PlateCarree(),
    cmap=cmap, norm=norm, shading="auto"
)
axes[1].scatter(
    sig_lon, sig_lat,
    transform=ccrs.PlateCarree(),
    s=MSIZE, c="k", alpha=ALPHA, marker="."
)
fig.colorbar(pcm1, ax=axes[1], ticks=MONTHS, label="Month")
axes[1].set_title("Wind stress magnitude |τ|: Month of Minimum")
axes[1].coastlines()

pcm2 = axes[2].pcolormesh(
    lon, lat, hblt_min_month,
    transform=ccrs.PlateCarree(),
    cmap=cmap, norm=norm, shading="auto"
)
axes[2].scatter(
    sig_lon, sig_lat,
    transform=ccrs.PlateCarree(),
    s=MSIZE, c="k", alpha=ALPHA, marker="."
)
fig.colorbar(pcm2, ax=axes[2], ticks=MONTHS, label="Month")
axes[2].set_title("HBLT: Month of Minimum")
axes[2].coastlines()

plt.show()



In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def month_of_max(da):
    mon_clim = da.groupby("time.month").mean("time")
    valid = mon_clim.notnull().any("month")
    out = mon_clim.fillna(-np.inf).argmax("month")
    return (out + 1).where(valid)

def wrap_month(m):
    return ((m - 1) % 12) + 1

def seasonal_ts_by_year(da, months):
    da_sel = da.where(da["time.month"].isin(months), drop=True)
    return da_sel.groupby("time.year").mean("time")

# ============================================================
# Load data (surface)
# ============================================================
da_pp   = select_surface(cfgs["photoC_TOT"].A_F_REF1_ds)
da_taux = select_surface(cfgs["TAUX"].A_F_REF1_ds)
da_tauy = select_surface(cfgs["TAUY"].A_F_REF1_ds)
da_q    = select_surface(cfgs["Q"].A_F_REF1_ds)
da_hblt = select_surface(cfgs["HBLT"].A_F_REF1_ds)

# ============================================================
# Derived
# ============================================================
da_wsm = np.sqrt(da_taux**2 + da_tauy**2)

# ============================================================
# PP peak month
# ============================================================
pp_peak_month = month_of_max(da_pp)

# ============================================================
# Correlation maps
# ============================================================
corr_wsm  = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_hblt = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_taux = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_tauy = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_q    = xr.full_like(pp_peak_month, np.nan, dtype=float)

for m in range(1, 13):

    # PP: peak & peak+1
    pp_months = [m, wrap_month(m + 1)]
    pp_ts = seasonal_ts_by_year(da_pp, pp_months)

    # WSM (lag: m-1, m-2)
    lag_months = [wrap_month(m - 1), wrap_month(m - 2)]

    wsm_ts = seasonal_ts_by_year(da_wsm, lag_months)
    pp_ts1, wsm_ts = xr.align(pp_ts, wsm_ts, join="inner")
    r_wsm = xr.corr(pp_ts1, wsm_ts, dim="year")

    hblt_ts = seasonal_ts_by_year(da_hblt, lag_months)
    pp_ts1b, hblt_ts = xr.align(pp_ts, hblt_ts, join="inner")
    r_hblt = xr.corr(pp_ts1b, hblt_ts, dim="year")

    # TAUX (no lag)
    taux_ts = seasonal_ts_by_year(da_taux, pp_months)
    pp_ts2, taux_ts = xr.align(pp_ts, taux_ts, join="inner")
    r_taux = xr.corr(pp_ts2, taux_ts, dim="year")

    # TAUY (no lag)
    tauy_ts = seasonal_ts_by_year(da_tauy, pp_months)
    pp_ts3, tauy_ts = xr.align(pp_ts, tauy_ts, join="inner")
    r_tauy = xr.corr(pp_ts3, tauy_ts, dim="year")

    # Q (no lag)
    q_ts = seasonal_ts_by_year(da_q, pp_months)
    pp_ts4, q_ts = xr.align(pp_ts, q_ts, join="inner")
    r_q = xr.corr(pp_ts4, q_ts, dim="year")

    corr_wsm  = xr.where(pp_peak_month == m, r_wsm,  corr_wsm)
    corr_hblt = xr.where(pp_peak_month == m, r_hblt, corr_hblt)
    corr_taux = xr.where(pp_peak_month == m, r_taux, corr_taux)
    corr_tauy = xr.where(pp_peak_month == m, r_tauy, corr_tauy)
    corr_q    = xr.where(pp_peak_month == m, r_q,    corr_q)

# ============================================================
# Plot: 2 columns × 3 rows + corr_DIFF dots
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(14, 12),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

corrs = [
    corr_wsm,  corr_hblt,
    corr_taux, corr_tauy,
    corr_q,    None
]

titles = [
    "PP vs |τ| (lag: peak−1, −2)",
    "PP vs HBLT (lag: peak−1, −2)",
    "PP vs TAUX (no lag)",
    "PP vs TAUY (no lag)",
    "PP vs Q (no lag)",
    ""
]

# -----------------------------
# DOT settings (USE corr_DIFF)
# -----------------------------
THRESH = 0.4
SUBSAMPLE = 4

lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

for ax, corr, title in zip(axes.flat, corrs, titles):

    if corr is None:
        ax.axis("off")
        continue

    pcm = ax.pcolormesh(
        lon, lat, corr,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-1, vmax=1, shading="auto"
    )

    # ==========================================
    # DOT: corr_DIFF > THRESH (precomputed!)
    # ==========================================
    mask_sig = corr_DIFF.values > THRESH

    mask_sub = np.zeros_like(mask_sig, dtype=bool)
    mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True
    mask_plot = mask_sig & mask_sub

    ax.scatter(
        LON[mask_plot], LAT[mask_plot],
        transform=ccrs.PlateCarree(),
        s=6, c="k", alpha=0.4, marker="."
    )

    ax.coastlines(linewidth=0.7)
    ax.set_global()
    ax.set_title(title)

cbar = fig.colorbar(pcm, ax=axes, shrink=0.9)
cbar.set_label("Correlation")

plt.show()



In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def month_of_max(da):
    mon_clim = da.groupby("time.month").mean("time")
    valid = mon_clim.notnull().any("month")
    out = mon_clim.fillna(-np.inf).argmax("month")
    return (out + 1).where(valid)

def wrap_month(m):
    return ((m - 1) % 12) + 1

def seasonal_ts_by_year(da, months):
    da_sel = da.where(da["time.month"].isin(months), drop=True)
    return da_sel.groupby("time.year").mean("time")

# ============================================================
# Load data (surface)
# ============================================================
da_pp   = select_surface(cfgs["photoC_TOT"].A_F_REF1_ds)
da_taux = select_surface(cfgs["TAUX"].A_F_REF1_ds)
da_tauy = select_surface(cfgs["TAUY"].A_F_REF1_ds)
da_n2   = select_surface(cfgs["N2"].A_F_REF1_ds)
da_hblt = select_surface(cfgs["HBLT"].A_F_REF1_ds)

# ============================================================
# Derived
# ============================================================
da_wsm = np.sqrt(da_taux**2 + da_tauy**2)

# ============================================================
# PP peak month
# ============================================================
pp_peak_month = month_of_max(da_pp)

# ============================================================
# Correlation maps
# ============================================================
corr_wsm       = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_hblt_lag  = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_taux      = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_tauy      = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_n2        = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_hblt_nl   = xr.full_like(pp_peak_month, np.nan, dtype=float)

for m in range(1, 13):

    # PP: peak & peak+1
    pp_months = [m, wrap_month(m + 1)]
    pp_ts = seasonal_ts_by_year(da_pp, pp_months)

    # lag months
    lag_months = [wrap_month(m - 1), wrap_month(m - 2)]

    # WSM (lag)
    wsm_ts = seasonal_ts_by_year(da_wsm, lag_months)
    pp1, wsm_ts = xr.align(pp_ts, wsm_ts, join="inner")
    r_wsm = xr.corr(pp1, wsm_ts, dim="year")

    # HBLT (lag)
    hblt_lag_ts = seasonal_ts_by_year(da_hblt, lag_months)
    pp1b, hblt_lag_ts = xr.align(pp_ts, hblt_lag_ts, join="inner")
    r_hblt_lag = xr.corr(pp1b, hblt_lag_ts, dim="year")

    # TAUX (no lag)
    taux_ts = seasonal_ts_by_year(da_taux, pp_months)
    pp2, taux_ts = xr.align(pp_ts, taux_ts, join="inner")
    r_taux = xr.corr(pp2, taux_ts, dim="year")

    # TAUY (no lag)
    tauy_ts = seasonal_ts_by_year(da_tauy, pp_months)
    pp3, tauy_ts = xr.align(pp_ts, tauy_ts, join="inner")
    r_tauy = xr.corr(pp3, tauy_ts, dim="year")

    # N2 (no lag)
    n2_ts = seasonal_ts_by_year(da_n2, pp_months)
    pp4, n2_ts = xr.align(pp_ts, n2_ts, join="inner")
    r_n2 = xr.corr(pp4, n2_ts, dim="year")

    # HBLT (no lag)  ← ★ 추가된 부분
    hblt_nl_ts = seasonal_ts_by_year(da_hblt, pp_months)
    pp5, hblt_nl_ts = xr.align(pp_ts, hblt_nl_ts, join="inner")
    r_hblt_nl = xr.corr(pp5, hblt_nl_ts, dim="year")

    corr_wsm      = xr.where(pp_peak_month == m, r_wsm,      corr_wsm)
    corr_hblt_lag = xr.where(pp_peak_month == m, r_hblt_lag, corr_hblt_lag)
    corr_taux     = xr.where(pp_peak_month == m, r_taux,     corr_taux)
    corr_tauy     = xr.where(pp_peak_month == m, r_tauy,     corr_tauy)
    corr_n2       = xr.where(pp_peak_month == m, r_n2,       corr_n2)
    corr_hblt_nl  = xr.where(pp_peak_month == m, r_hblt_nl,  corr_hblt_nl)

# ============================================================
# Plot: 2 columns × 3 rows
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(14, 12),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

corrs = [
    corr_wsm,       corr_hblt_lag,
    corr_taux,      corr_tauy,
    corr_n2,        corr_hblt_nl
]

titles = [
    "PP vs |τ| (lag: peak−1, −2)",
    "PP vs HBLT (lag: peak−1, −2)",
    "PP vs TAUX (no lag)",
    "PP vs TAUY (no lag)",
    "PP vs N2 (no lag)",
    "PP vs HBLT (no lag)"
]

# DOT settings
THRESH = 0.4
SUBSAMPLE = 4

lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

for ax, corr, title in zip(axes.flat, corrs, titles):

    pcm = ax.pcolormesh(
        lon, lat, corr,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-1, vmax=1, shading="auto"
    )

    mask_sig = corr_DIFF.values > THRESH
    mask_sub = np.zeros_like(mask_sig, dtype=bool)
    mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True

    ax.scatter(
        LON[mask_sig & mask_sub], LAT[mask_sig & mask_sub],
        transform=ccrs.PlateCarree(),
        s=6, c="k", alpha=0.4, marker="."
    )

    ax.coastlines(linewidth=0.7)
    ax.set_global()
    ax.set_title(title)

cbar = fig.colorbar(pcm, ax=axes, shrink=0.9)
cbar.set_label("Correlation")

plt.show()


In [ ]:
# Apr, May only

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def month_of_max(da):
    mon_clim = da.groupby("time.month").mean("time")
    valid = mon_clim.notnull().any("month")
    out = mon_clim.fillna(-np.inf).argmax("month")
    return (out + 1).where(valid)

def wrap_month(m):
    return ((m - 1) % 12) + 1

def seasonal_ts_by_year(da, months):
    da_sel = da.where(da["time.month"].isin(months), drop=True)
    return da_sel.groupby("time.year").mean("time")

# ============================================================
# Load data (surface)
# ============================================================
da_pp   = select_surface(cfgs["photoC_TOT"].A_F_REF1_ds)
da_taux = select_surface(cfgs["TAUX"].A_F_REF1_ds)
da_tauy = select_surface(cfgs["TAUY"].A_F_REF1_ds)
da_q    = select_surface(cfgs["Q"].A_F_REF1_ds)
da_hblt = select_surface(cfgs["HBLT"].A_F_REF1_ds)

# ============================================================
# Derived
# ============================================================
da_wsm = np.sqrt(da_taux**2 + da_tauy**2)

# ============================================================
# PP peak month
# ============================================================
pp_peak_month = month_of_max(da_pp)

# ============================================================
# Correlation maps
# ============================================================
corr_wsm  = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_hblt = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_taux = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_tauy = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_q    = xr.full_like(pp_peak_month, np.nan, dtype=float)

for m in range(1, 13):

    # PP: peak & peak+1
    pp_months = [m, wrap_month(m + 1)]
    pp_ts = seasonal_ts_by_year(da_pp, pp_months)

    # WSM / HBLT (lag: m-1, m-2)
    lag_months = [wrap_month(m - 1), wrap_month(m - 2)]

    wsm_ts = seasonal_ts_by_year(da_wsm, lag_months)
    pp_ts1, wsm_ts = xr.align(pp_ts, wsm_ts, join="inner")
    r_wsm = xr.corr(pp_ts1, wsm_ts, dim="year")

    hblt_ts = seasonal_ts_by_year(da_hblt, lag_months)
    pp_ts1b, hblt_ts = xr.align(pp_ts, hblt_ts, join="inner")
    r_hblt = xr.corr(pp_ts1b, hblt_ts, dim="year")

    # TAUX (no lag)
    taux_ts = seasonal_ts_by_year(da_taux, pp_months)
    pp_ts2, taux_ts = xr.align(pp_ts, taux_ts, join="inner")
    r_taux = xr.corr(pp_ts2, taux_ts, dim="year")

    # TAUY (no lag)
    tauy_ts = seasonal_ts_by_year(da_tauy, pp_months)
    pp_ts3, tauy_ts = xr.align(pp_ts, tauy_ts, join="inner")
    r_tauy = xr.corr(pp_ts3, tauy_ts, dim="year")

    # Q (no lag)
    q_ts = seasonal_ts_by_year(da_q, pp_months)
    pp_ts4, q_ts = xr.align(pp_ts, q_ts, join="inner")
    r_q = xr.corr(pp_ts4, q_ts, dim="year")

    corr_wsm  = xr.where(pp_peak_month == m, r_wsm,  corr_wsm)
    corr_hblt = xr.where(pp_peak_month == m, r_hblt, corr_hblt)
    corr_taux = xr.where(pp_peak_month == m, r_taux, corr_taux)
    corr_tauy = xr.where(pp_peak_month == m, r_tauy, corr_tauy)
    corr_q    = xr.where(pp_peak_month == m, r_q,    corr_q)

# ============================================================
# Mask: only PP peak in April or May
# ============================================================
mask_apr_may = (pp_peak_month == 4) | (pp_peak_month == 5)

corr_wsm  = corr_wsm.where(mask_apr_may)
corr_hblt = corr_hblt.where(mask_apr_may)
corr_taux = corr_taux.where(mask_apr_may)
corr_tauy = corr_tauy.where(mask_apr_may)
corr_q    = corr_q.where(mask_apr_may)

# ============================================================
# Plot: 2 columns × 3 rows + corr_DIFF dots
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(14, 12),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

corrs = [
    corr_wsm,  corr_hblt,
    corr_taux, corr_tauy,
    corr_q,    None
]

titles = [
    "PP vs |τ| (lag: peak−1, −2)",
    "PP vs HBLT (lag: peak−1, −2)",
    "PP vs TAUX (no lag)",
    "PP vs TAUY (no lag)",
    "PP vs Q (no lag)",
    ""
]

# DOT settings (precomputed corr_DIFF)
THRESH = 0.4
SUBSAMPLE = 4

lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

for ax, corr, title in zip(axes.flat, corrs, titles):

    if corr is None:
        ax.axis("off")
        continue

    pcm = ax.pcolormesh(
        lon, lat, corr,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-1, vmax=1, shading="auto"
    )

    mask_sig = (corr_DIFF.values > THRESH) & mask_apr_may.values

    mask_sub = np.zeros_like(mask_sig, dtype=bool)
    mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True
    mask_plot = mask_sig & mask_sub

    ax.scatter(
        LON[mask_plot], LAT[mask_plot],
        transform=ccrs.PlateCarree(),
        s=6, c="k", alpha=0.4, marker="."
    )

    ax.coastlines(linewidth=0.7)
    ax.set_global()
    ax.set_title(title)

cbar = fig.colorbar(pcm, ax=axes, shrink=0.9)
cbar.set_label("Correlation")

plt.show()


In [ ]:
# May only

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def month_of_max(da):
    mon_clim = da.groupby("time.month").mean("time")
    valid = mon_clim.notnull().any("month")
    out = mon_clim.fillna(-np.inf).argmax("month")
    return (out + 1).where(valid)

def wrap_month(m):
    return ((m - 1) % 12) + 1

def seasonal_ts_by_year(da, months):
    da_sel = da.where(da["time.month"].isin(months), drop=True)
    return da_sel.groupby("time.year").mean("time")

# ============================================================
# Load data (surface)
# ============================================================
da_pp   = select_surface(cfgs["photoC_TOT"].A_F_REF1_ds)
da_taux = select_surface(cfgs["TAUX"].A_F_REF1_ds)
da_tauy = select_surface(cfgs["TAUY"].A_F_REF1_ds)
da_q    = select_surface(cfgs["Q"].A_F_REF1_ds)
da_hblt = select_surface(cfgs["HBLT"].A_F_REF1_ds)

# ============================================================
# Derived
# ============================================================
da_wsm = np.sqrt(da_taux**2 + da_tauy**2)

# ============================================================
# PP peak month
# ============================================================
pp_peak_month = month_of_max(da_pp)

# ============================================================
# Correlation maps
# ============================================================
corr_wsm  = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_hblt = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_taux = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_tauy = xr.full_like(pp_peak_month, np.nan, dtype=float)
corr_q    = xr.full_like(pp_peak_month, np.nan, dtype=float)

for m in range(1, 13):

    # PP: peak & peak+1
    pp_months = [m, wrap_month(m + 1)]
    pp_ts = seasonal_ts_by_year(da_pp, pp_months)

    # WSM / HBLT (lag: m-1, m-2)
    lag_months = [wrap_month(m - 1), wrap_month(m - 2)]

    wsm_ts = seasonal_ts_by_year(da_wsm, lag_months)
    pp_ts1, wsm_ts = xr.align(pp_ts, wsm_ts, join="inner")
    r_wsm = xr.corr(pp_ts1, wsm_ts, dim="year")

    hblt_ts = seasonal_ts_by_year(da_hblt, lag_months)
    pp_ts1b, hblt_ts = xr.align(pp_ts, hblt_ts, join="inner")
    r_hblt = xr.corr(pp_ts1b, hblt_ts, dim="year")

    # TAUX (no lag)
    taux_ts = seasonal_ts_by_year(da_taux, pp_months)
    pp_ts2, taux_ts = xr.align(pp_ts, taux_ts, join="inner")
    r_taux = xr.corr(pp_ts2, taux_ts, dim="year")

    # TAUY (no lag)
    tauy_ts = seasonal_ts_by_year(da_tauy, pp_months)
    pp_ts3, tauy_ts = xr.align(pp_ts, tauy_ts, join="inner")
    r_tauy = xr.corr(pp_ts3, tauy_ts, dim="year")

    # Q (no lag)
    q_ts = seasonal_ts_by_year(da_q, pp_months)
    pp_ts4, q_ts = xr.align(pp_ts, q_ts, join="inner")
    r_q = xr.corr(pp_ts4, q_ts, dim="year")

    corr_wsm  = xr.where(pp_peak_month == m, r_wsm,  corr_wsm)
    corr_hblt = xr.where(pp_peak_month == m, r_hblt, corr_hblt)
    corr_taux = xr.where(pp_peak_month == m, r_taux, corr_taux)
    corr_tauy = xr.where(pp_peak_month == m, r_tauy, corr_tauy)
    corr_q    = xr.where(pp_peak_month == m, r_q,    corr_q)

# ============================================================
# Mask: only PP peak in April or May
# ============================================================
mask_apr_may = (pp_peak_month == 5)

corr_wsm  = corr_wsm.where(mask_apr_may)
corr_hblt = corr_hblt.where(mask_apr_may)
corr_taux = corr_taux.where(mask_apr_may)
corr_tauy = corr_tauy.where(mask_apr_may)
corr_q    = corr_q.where(mask_apr_may)

# ============================================================
# Plot: 2 columns × 3 rows + corr_DIFF dots
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(14, 12),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

corrs = [
    corr_wsm,  corr_hblt,
    corr_taux, corr_tauy,
    corr_q,    None
]

titles = [
    "PP vs |τ| (lag: peak−1, −2)",
    "PP vs HBLT (lag: peak−1, −2)",
    "PP vs TAUX (no lag)",
    "PP vs TAUY (no lag)",
    "PP vs Q (no lag)",
    ""
]

# DOT settings (precomputed corr_DIFF)
THRESH = 0.4
SUBSAMPLE = 4

lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

for ax, corr, title in zip(axes.flat, corrs, titles):

    if corr is None:
        ax.axis("off")
        continue

    pcm = ax.pcolormesh(
        lon, lat, corr,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r", vmin=-1, vmax=1, shading="auto"
    )

    mask_sig = (corr_DIFF.values > THRESH) & mask_apr_may.values

    mask_sub = np.zeros_like(mask_sig, dtype=bool)
    mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True
    mask_plot = mask_sig & mask_sub

    ax.scatter(
        LON[mask_plot], LAT[mask_plot],
        transform=ccrs.PlateCarree(),
        s=6, c="k", alpha=0.4, marker="."
    )

    ax.coastlines(linewidth=0.7)
    ax.set_global()
    ax.set_title(title)

cbar = fig.colorbar(pcm, ax=axes, shrink=0.9)
cbar.set_label("Correlation")

plt.show()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# ============================================================
# Helpers
# ============================================================
def select_surface(da):
    if "z_t" in da.dims:
        return da.isel(z_t=0)
    elif "z_t_150m" in da.dims:
        return da.isel(z_t_150m=0)
    elif "z_w_top" in da.dims:
        return da.isel(z_w_top=0)
    else:
        return da

def deseason(da):
    clim = da.groupby("time.month").mean("time")
    return da.groupby("time.month") - clim

# ============================================================
# Load data (surface + deseason)
# ============================================================
da_pp   = deseason(select_surface(cfgs["photoC_TOT"].A_F_REF1_ds))
da_hblt = deseason(select_surface(cfgs["HBLT"].A_F_REF1_ds))
da_taux = deseason(select_surface(cfgs["TAUX"].A_F_REF1_ds))
da_tauy = deseason(select_surface(cfgs["TAUY"].A_F_REF1_ds))
da_q    = deseason(select_surface(cfgs["Q"].A_F_REF1_ds))

# ============================================================
# Derived
# ============================================================
da_wsm = np.sqrt(da_taux**2 + da_tauy**2)

# ============================================================
# Correlations (ALL months, deseasoned)
# ============================================================
# 1. HBLT leads PP by 1 month
pp0, hblt1 = xr.align(da_pp, da_hblt.shift(time=1), join="inner")
corr_hblt = xr.corr(pp0, hblt1, dim="time")

# 2. WSM leads PP by 1 month
pp1, wsm1 = xr.align(da_pp, da_wsm.shift(time=1), join="inner")
corr_wsm = xr.corr(pp1, wsm1, dim="time")

# 3. TAUX same month
pp2, taux2 = xr.align(da_pp, da_taux, join="inner")
corr_taux = xr.corr(pp2, taux2, dim="time")

# 4. TAUY same month
pp3, tauy3 = xr.align(da_pp, da_tauy, join="inner")
corr_tauy = xr.corr(pp3, tauy3, dim="time")

# 5. Q same month
pp4, q4 = xr.align(da_pp, da_q, join="inner")
corr_q = xr.corr(pp4, q4, dim="time")

# ============================================================
# Mask (corr_WIND − corr_xALL)
# ============================================================
THRESH = 0.4
SUBSAMPLE = 4

mask_sig = (corr_DIFF > THRESH)

lon = da_pp["lon"].values
lat = da_pp["lat"].values
LON, LAT = np.meshgrid(lon, lat)

mask_sub = np.zeros_like(mask_sig.values, dtype=bool)
mask_sub[::SUBSAMPLE, ::SUBSAMPLE] = True
mask_plot = mask_sig.values & mask_sub

sig_lon = LON[mask_plot]
sig_lat = LAT[mask_plot]

# ============================================================
# Plot: 5 panels, 2 columns
# ============================================================
proj = ccrs.PlateCarree(central_longitude=205)

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(12, 10),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

axes = axes.flatten()

corrs = [
    corr_hblt,
    corr_wsm,
    corr_taux,
    corr_tauy,
    corr_q,
]

titles = [
    "PP vs HBLT (HBLT leads by 1 month, deseasoned)",
    "PP vs |τ| (WSM leads by 1 month, deseasoned)",
    "PP vs TAUX (same month, deseasoned)",
    "PP vs TAUY (same month, deseasoned)",
    "PP vs Q (same month, deseasoned)",
]

for ax, corr, title in zip(axes, corrs, titles):

    pcm = ax.pcolormesh(
        lon, lat, corr,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        vmin=-1, vmax=1,
        shading="auto"
    )

    ax.scatter(
        sig_lon, sig_lat,
        transform=ccrs.PlateCarree(),
        s=6, c="k", alpha=0.4, marker="."
    )

    ax.coastlines(linewidth=0.7)
    ax.set_global()
    ax.set_title(title)

# remove unused last panel if any
for ax in axes[len(corrs):]:
    ax.remove()

cbar = fig.colorbar(pcm, ax=axes[:len(corrs)], shrink=0.9)
cbar.set_label("Correlation")

plt.show()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cmocean

# ==========================================================
# 1. Load DataArrays (★ Dataset → DataArray로 수정)
# ==========================================================
da1 = cfgs['photoC_TOT'].A_F_REF1_ds
da2 = cfgs['photoC_TOT'].A_F_REF2_ds
# da3 = cfgs['photoC_TOT'].A_F_DUST_ds
da4 = cfgs['photoC_TOT'].A_F_WIND_ds
# da5 = cfgs['photoC_TOT'].A_F_SOLAR_ds
# da6 = cfgs['photoC_TOT'].A_F_WDS_ds

# # 안전하게 time 정렬
# da1, da2 = xr.align(da1, da2, join='inner')
# da1, da3 = xr.align(da1, da3, join='inner')


# ==========================================================
# 2. Remove seasonal cycle (monthly climatology)
# ==========================================================
da1_anom = da1.groupby('time.month') - da1.groupby('time.month').mean('time')
da2_anom = da2.groupby('time.month') - da2.groupby('time.month').mean('time')
# da3_anom = da3.groupby('time.month') - da3.groupby('time.month').mean('time')
da4_anom = da4.groupby('time.month') - da4.groupby('time.month').mean('time')
# da5_anom = da5.groupby('time.month') - da5.groupby('time.month').mean('time')
# da6_anom = da6.groupby('time.month') - da6.groupby('time.month').mean('time')

# # ==========================================================
# # 3. Remove linear trend (along time)
# # ==========================================================
# def detrend(da, dim='time'):
#     p = da.polyfit(dim=dim, deg=1)
#     trend = xr.polyval(da[dim], p.polyfit_coefficients)
#     return da - trend

# da1_dt = detrend(da1_anom)
# da2_dt = detrend(da2_anom)

# ==========================================================
# 4. Correlations (surface only)
# ==========================================================
# corr_raw = xr.corr(da1, da2, dim='time').isel(z_t=0)
corr_ds2  = xr.corr(da1_anom, da2_anom, dim='time').isel(z_t_150m=0)
# corr_dt  = xr.corr(da1_dt, da2_dt, dim='time').isel(z_t=0)
# corr_ds3  = xr.corr(da1_anom, da3_anom, dim='time').isel(z_t_150m=0)
corr_ds4  = xr.corr(da1_anom, da4_anom, dim='time').isel(z_t_150m=0)
# corr_ds5  = xr.corr(da1_anom, da5_anom, dim='time').isel(z_t_150m=0)
# corr_ds6  = xr.corr(da1_anom, da6_anom, dim='time').isel(z_t_150m=0)
# corr_ds7  = xr.corr(da1_anom, da7_anom, dim='time').isel(z_t_150m=0)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs




# ==========================================================
# Settings
# ==========================================================
vmin_corr  = -1.0
vmax_corr  =  1.0
vmin_diff  = -0.8
vmax_diff  =  0.8

MARKER = "*"     # 원하는 마커: "*", "o", "^", "x" 등
MSIZE  = 120
MCOLOR = "blue"

# ==========================================================
# Projection (center longitude = 205)
# ==========================================================
proj = ccrs.PlateCarree(central_longitude=205)

# ==========================================================
# Figure: 1 row × 2 columns
# ==========================================================
fig, axes = plt.subplots(
    nrows=1, ncols=2,
    figsize=(12, 6),
    constrained_layout=True,
    subplot_kw=dict(projection=proj)
)

# ==========================================================
# Panel 1: corr_ds2
# ==========================================================
ax = axes[0]
pcm1 = ax.pcolormesh(
    da1.lon, da1.lat, corr_ds2,
    cmap='RdBu_r',
    vmin=vmin_corr, vmax=vmax_corr,
    transform=ccrs.PlateCarree(),
    shading='auto'
)
ax.set_title("corr_ds2")
ax.coastlines(linewidth=0.7)
ax.set_global()

# mark points (symbol only)
for lon0, lat0 in points:
    ax.scatter(
        lon0, lat0,
        transform=ccrs.PlateCarree(),
        marker=MARKER,
        s=MSIZE,
        c=MCOLOR,
        zorder=5
    )

# ==========================================================
# Panel 2: corr_ds4 − corr_ds2
# ==========================================================
ax = axes[1]
pcm2 = ax.pcolormesh(
    da1.lon, da1.lat, corr_ds4 - corr_ds2,
    cmap='PuOr_r',
    vmin=vmin_diff, vmax=vmax_diff,
    transform=ccrs.PlateCarree(),
    shading='auto'
)
ax.set_title("corr_ds4 − corr_ds2")
ax.coastlines(linewidth=0.7)
ax.set_global()

# mark points (symbol only)
for lon0, lat0 in points:
    ax.scatter(
        lon0, lat0,
        transform=ccrs.PlateCarree(),
        marker=MARKER,
        s=MSIZE,
        c=MCOLOR,
        zorder=5
    )

# ==========================================================
# Colorbars (one per panel)
# ==========================================================
cbar1 = fig.colorbar(pcm1, ax=axes[0], shrink=0.85)
cbar1.set_label("Correlation")

cbar2 = fig.colorbar(pcm2, ax=axes[1], shrink=0.85, extend='both')
cbar2.set_label("Correlation difference")

plt.show()
